# MiniMax H3 完全版（ファイル名・プロンプト入り）

**ココナラの10秒モーション広告はセル10を実行する。** セル8（R2V）とセル9（アクション）は別用途。

セル10に最初から入っている値:

- 画像: `Image 1.jpg`（Drive の `input/` に、和室の女の子静止画をこの名前で置く）
- キャンバス: `768×864`（8:9。A100 40GB で 1024×1152 は OOM したので既定を下げた。足りなければ自動で 512×576）
- 長さ: 10秒
- プロンプト: Shot 1〜10（ペン→線→文字→カード→CTA）がセル内に入っている

手順: セル1 → 2 → 3（`MODE=both`） → 4 → writefile 3本 → セル8a → **セル10**

動画内にアフィURLは出さない。押せる場所はプロフィール。画面上は「広告」+「無料で始める」。


## セル8：参照動画のモーション × 参照画像の人物

このノートの R2V は **ComfyUI の MiniMaxH3ReferenceToVideo（ref2va）** で動かす。

### うまく動かなかった原因（修正済み）

1. **セル8先頭が `print(= * 60)` で SyntaxError** → セル自体が実行できなかった
2. **動画配線に失敗すると `use_videos=False` へ黙ってフォールバック** → 人物画像だけの生成になり、モーション転写したように見えない
3. **セル3の既定 MODE が `t2v_i2v`** → R2V 必須の `ref2va` が入らない
4. **線画化（STRIP_VIDEO_IDENTITY=True）が既定ON** → H3 は実写クリップからモーションを読む。Canny 線画だと動きが消えることが多い
5. **LoadImage / VHS がファイル名だけ** → サブフォルダの素材を拾えない


6. **14秒 + `REF_IMAGE_SIZE=max` + 参照動画は A100 40GB で OOM** → 40GB では約6秒に落とす。**A100 ハイメモリ（80GB VRAM）なら 14秒+max をそのまま通す**。Colab の High-RAM は CPU メモリなので、セル1の `VRAM GiB` が 40 のままなら足りない

### いまの正しい使い方

- ランタイムは **A100 High Memory / 80GB GPU**。セル1で `VRAM GiB: 80` 前後を確認
- セル3 の MODE は **both**（または r2v）
- セル8: IMAGE_FILES = 人物、VIDEO_FILES = 動き。`DURATION_S=14`、`REF_IMAGE_SIZE=max`。`STRIP_VIDEO_IDENTITY` は顔が混ざるときだけ True
- 動画なしフォールバックはしない（失敗したらエラーで止める）
- **激しいアクションの「動き完全コピー」は H3 では足りない** → **セル9（Wan 2.2 Animate Mix）**。骨格（DWPose）で動きを固定し、元映像のカメラのまま人物だけ差し替える


In [ ]:
#@title セル1：Google Drive 接続 + 準備チェック
# ##############################################################################
print("=" * 60)
print(" セル1：Google Drive 接続 + 準備")
print("=" * 60)

from google.colab import drive
import os

# ---------- 設定（必要なら変更）----------
# Drive 上の保存先（初回に自動作成）
DRIVE_ROOT = "/content/drive/MyDrive/minimax-h3-comfyui"  #@param {type:"string"}
COMFY_DIR = "/content/ComfyUI"
# ----------------------------------------

drive.mount("/content/drive")

# Drive 側ディレクトリ
DRIVE_MODELS = f"{DRIVE_ROOT}/models"
for sub in ["diffusion_models", "text_encoders", "vae", "checkpoints", "loras", "controlnet", "clip_vision"]:
    os.makedirs(f"{DRIVE_MODELS}/{sub}", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/output", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/input", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/workflows", exist_ok=True)

print("Drive 保存先:", DRIVE_ROOT)
print("モデル置き場:", DRIVE_MODELS)

# 容量表示
print("\n--- Colab (/content) ---")
!df -h /content | tail -1
print("--- Google Drive (目安) ---")
!df -h /content/drive 2>/dev/null | tail -1 || echo "(Drive の df は環境により出ないことがあります)"

!nvidia-smi -L
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram = props.total_memory / 1024**3
    host = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
    print("GPU:", torch.cuda.get_device_name(0), "VRAM GiB:", round(vram, 1), "host RAM GiB:", round(host, 0))
    if vram >= 70:
        print("判定: 80GB クラス（A100 ハイメモリGPU）。14秒 + REF_IMAGE_SIZE=max を通します。")
    elif vram >= 32 and host >= 50:
        print("判定: GPU は 40GB。host RAM だけ多い（High-RAM）です。前回の 14秒 R2V OOM はこれでは解消しません。")
        print("      ランタイム → タイプを変更 → A100 High Memory（80GB GPU）を選んでください。")
    elif vram >= 32:
        print("判定: A100 40GB。14秒+max は自動で約6秒に落とします。")
    else:
        print("判定: VRAM 不足。A100 を選んでください。")
else:
    print("⚠ GPU OFF → ランタイム → ランタイムのタイプを変更 → A100 High Memory")

# 後続セル用にパスをファイルへ保存
with open("/content/h3_paths.env", "w") as f:
    f.write(f"DRIVE_ROOT={DRIVE_ROOT}\n")
    f.write(f"DRIVE_MODELS={DRIVE_MODELS}\n")
    f.write(f"COMFY_DIR={COMFY_DIR}\n")

print("\nOK → 次は【セル2】")


# ##############################################################################


In [ ]:
#@title セル2：ComfyUI インストール + Drive の models を接続
# ##############################################################################
print("=" * 60)
print(" セル2：ComfyUI + Drive リンク")
print("=" * 60)

import os

# パス読み込み
env = {}
with open("/content/h3_paths.env") as f:
    for line in f:
        k, v = line.strip().split("=", 1)
        env[k] = v
DRIVE_ROOT = env["DRIVE_ROOT"]
DRIVE_MODELS = env["DRIVE_MODELS"]
COMFY_DIR = env["COMFY_DIR"]

# ComfyUI は /content に（軽くて速い）
if not os.path.exists(COMFY_DIR):
    !git clone --depth 1 https://github.com/Comfy-Org/ComfyUI.git {COMFY_DIR}
else:
    print("ComfyUI 既存 → pull")
    %cd {COMFY_DIR}
    !git pull --ff-only || true

%cd {COMFY_DIR}
!pip install -q -r requirements.txt

# ---------------------------------------------------------------------------
# 速度用 custom_nodes（よく考えて入れた推奨セット）
#
# 判断メモ:
#   - 公式 H3 ドキュメントが推すのは Sage Attention ≒ 最大~2倍・品質ほぼ維持
#   - SolAttn_triton は sparse attention。H3 向けに使われ、+15〜20% の報告あり
#     ただし README も「experimental」、初回 Triton コンパイルで遅い、
#     品質は tau 次第で落ちる。作者テストは主に 4090/5090
#   - 4090 ベンチでは「H3 Mem Eff Sage」単独の方が Sol 単独より速い例が多い
#   - 結論: Sage を本命で必須寄りに入れ、Sol は ON で入れておくが
#            ワークフローでは「まず Sage だけ → 足りなければ Sol 追加」
# ---------------------------------------------------------------------------
INSTALL_SPEED_NODES = True  #@param {type:"boolean"}
# Sol-Attn リポジトリも clone する（ノードは使うときだけグラフに足す）
INSTALL_SOLATTN = True  #@param {type:"boolean"}
# SageAttention pip（失敗しても Comfy 自体は動く）
INSTALL_SAGEATTENTION = True  #@param {type:"boolean"}

cn = f"{COMFY_DIR}/custom_nodes"
os.makedirs(cn, exist_ok=True)

def clone_or_pull(url, folder_name):
    path = os.path.join(cn, folder_name)
    if os.path.isdir(path):
        print(f"更新: {folder_name}")
        !git -C "{path}" pull --ff-only || true
    else:
        print(f"clone: {folder_name}")
        !git clone --depth 1 "{url}" "{path}"
    req = os.path.join(path, "requirements.txt")
    if os.path.isfile(req):
        !pip install -q -r "{req}" || true
    return path

if INSTALL_SPEED_NODES:
    print("\n=== 速度ノード導入 ===")
    # 1) KJNodes = Patch Sage / H3 Mem Eff Sage など（本命）
    clone_or_pull("https://github.com/kijai/ComfyUI-KJNodes.git", "ComfyUI-KJNodes")

    # 2) Sol-Attn Triton（任意加速・実験的）
    if INSTALL_SOLATTN:
        clone_or_pull(
            "https://github.com/kijai/ComfyUI-SolAttn_triton.git",
            "ComfyUI-SolAttn_triton",
        )
        # Triton: Colab Linux では通常 torch 同梱 or pip で入る
        !pip install -q -U triton || true
        print("SolAttn: 初回生成は kernel コンパイルで遅くなることがあります")

    # 2.5) Video Helper Suite = VHS_LoadVideo（R2V モーション）
    INSTALL_VHS = True
    if INSTALL_VHS:
        clone_or_pull(
            "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git",
            "ComfyUI-VideoHelperSuite",
        )

    # 2.6) DWPose = アクションの骨格ロック（セル9 Wan Animate）
    clone_or_pull(
        "https://github.com/Fannovel16/comfyui_controlnet_aux.git",
        "comfyui_controlnet_aux",
    )

    # 3) SageAttention 本体
    if INSTALL_SAGEATTENTION:
        print("SageAttention インストール試行...")
        # まず素直に pip。Colab の torch/cuda と合わない場合は失敗しても続行
        !pip install -q sageattention 2>/dev/null || \
         pip install -q git+https://github.com/thu-ml/SageAttention.git 2>/dev/null || \
         echo "SageAttention pip 失敗 → 後で wheel を合わせるか、KJNodes の一部機能のみで継続"

    try:
        import sageattention  # noqa: F401
        print("✓ sageattention import OK")
    except Exception as e:
        print("△ sageattention 未導入（標準 attention で動作。速度は落ちる）:", e)

    try:
        import triton  # noqa: F401
        print("✓ triton import OK:", getattr(triton, "__version__", "?"))
    except Exception as e:
        print("△ triton 未導入（SolAttn は動かない可能性）:", e)

    print("custom_nodes:")
    !ls -1 "{cn}"
else:
    print("速度ノードスキップ（INSTALL_SPEED_NODES=False）")

# --- models を Drive に接続（本体は Drive、Colab にはリンクだけ）---
def link_dir(link_path, target_path):
    """link_path → target_path のシンボリックリンク。既存は退避/削除して作り直す"""
    os.makedirs(target_path, exist_ok=True)
    if os.path.islink(link_path):
        os.remove(link_path)
    elif os.path.isdir(link_path):
        # 中身が空っぽい or リンクしたいのでリネーム退避
        bak = link_path + ".local_bak"
        if os.path.exists(bak):
            import shutil
            shutil.rmtree(bak, ignore_errors=True)
        os.rename(link_path, bak)
        print("  退避:", bak)
    elif os.path.exists(link_path):
        os.remove(link_path)
    os.symlink(target_path, link_path)
    print(f"  link: {link_path}  →  {target_path}")

print("\nDrive の models を ComfyUI に接続:")
models_root = f"{COMFY_DIR}/models"
os.makedirs(models_root, exist_ok=True)

# 主要サブフォルダを個別リンク（ComfyUI が他サブフォルダを作っても壊れにくい）
for sub in ["diffusion_models", "text_encoders", "vae", "checkpoints", "loras", "controlnet", "clip_vision"]:
    link_dir(f"{models_root}/{sub}", f"{DRIVE_MODELS}/{sub}")

# output / input も Drive へ（生成物が消えない）
link_dir(f"{COMFY_DIR}/output", f"{DRIVE_ROOT}/output")
link_dir(f"{COMFY_DIR}/input", f"{DRIVE_ROOT}/input")

# 公式 workflow を Drive と user 両方へ
wf_user = f"{COMFY_DIR}/user/default/workflows"
os.makedirs(wf_user, exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/workflows", exist_ok=True)
base = "https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main/templates"
for name in [
    "video_minimax_h3_t2v.json",
    "video_minimax_h3_i2v.json",
    "video_minimax_h3_r2v.json",
]:
    !wget -q -O "{DRIVE_ROOT}/workflows/{name}" "{base}/{name}"
    !cp -f "{DRIVE_ROOT}/workflows/{name}" "{wf_user}/{name}"
    print("workflow:", name)

print("\n接続確認:")
!ls -la {COMFY_DIR}/models/diffusion_models | head -5
!ls -la {COMFY_DIR}/output | head -3

print("\nOK → 次は【セル3】（Drive に無ければDL、あればスキップ）")


# ##############################################################################



In [ ]:
#@title セル3：モデルを Google Drive へ取得（あればスキップ）
# ##############################################################################
print("=" * 60)
print(" セル3：モデル → Google Drive")
print("=" * 60)
print(" Colab のディスクではなく Drive に保存します")
print(" R2V には ref2va が必要 → 既定 MODE=both")
print("=" * 60)

import os

env = {}
with open("/content/h3_paths.env") as f:
    for line in f:
        k, v = line.strip().split("=", 1)
        env[k] = v
DRIVE_MODELS = env["DRIVE_MODELS"]
DRIVE_ROOT = env["DRIVE_ROOT"]

HF = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main"

# モード: t2v_i2v だけなら約42GB / both だと +約21GB
MODE = "both"  #@param ["t2v_i2v", "r2v", "both"]
DIT_QUANT = "int8"  #@param ["int8", "fp8"]

if DIT_QUANT == "int8":
    fl2va = "minimax_h3_fl2va_pruned_int8_convrot.safetensors"
    ref2va = "minimax_h3_ref2va_pruned_int8_convrot.safetensors"
else:
    fl2va = "minimax_h3_fl2va_pruned_fp8_scaled.safetensors"
    ref2va = "minimax_h3_ref2va_pruned_fp8_scaled.safetensors"

files = [
    (f"text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
     f"{DRIVE_MODELS}/text_encoders"),
    (f"vae/minimax_h3_video_vae_fp16.safetensors",
     f"{DRIVE_MODELS}/vae"),
    (f"vae/minimax_h3_audio_vae_fp32.safetensors",
     f"{DRIVE_MODELS}/vae"),
]
if MODE in ("t2v_i2v", "both"):
    files.append((f"diffusion_models/{fl2va}", f"{DRIVE_MODELS}/diffusion_models"))
if MODE in ("r2v", "both"):
    files.append((f"diffusion_models/{ref2va}", f"{DRIVE_MODELS}/diffusion_models"))

MIN_OK = 1_000_000  # 1MB 未満は失敗扱い

for rel, folder in files:
    os.makedirs(folder, exist_ok=True)
    name = os.path.basename(rel)
    path = os.path.join(folder, name)
    if os.path.exists(path) and os.path.getsize(path) > MIN_OK:
        gb = os.path.getsize(path) / 1e9
        print(f"✓ Drive にあり（スキップ）: {name}  ({gb:.2f} GB)")
        continue
    url = f"{HF}/{rel}"
    print(f"↓ Drive へダウンロード: {name}")
    print(f"  保存先: {path}")
    # -c で再開可能（途中切断に強い）
    !wget -c --show-progress -O "{path}" "{url}"
    if not os.path.exists(path) or os.path.getsize(path) < MIN_OK:
        # 壊れたファイルを残さない
        if os.path.exists(path):
            os.remove(path)
        raise SystemExit(f"DL 失敗: {name}\nDrive の空き容量を確認してください（目安 45GB+）")
    print(f"✓ 完了: {name}  ({os.path.getsize(path)/1e9:.2f} GB)")

print("\n--- Drive 上の MiniMax 関連ファイル ---")
!find "{DRIVE_MODELS}" -type f \( -name "*minimax*" -o -name "*qwen3vl*" \) -printf "%s\t%p\n" 2>/dev/null | awk '{printf "%.2f GB\t%s\n", $1/1e9, $2}'

print("\nColab ディスク:")
!df -h /content | tail -1
print("\nOK → 次は【セル4】起動")


# ##############################################################################


In [ ]:
#@title セル4：起動して UI を開く（loca.lt / colab.dev は使わない）
# ##############################################################################
print("=" * 60)
print(" セル4：ComfyUI 起動")
print("=" * 60)
print(" 開くのは loca.lt の URL だけ（colab.dev は開かない）")
print("=" * 60)

import os
import re
import shutil
import subprocess
import sys
import time
import urllib.request

env = {}
with open("/content/h3_paths.env") as f:
    for line in f:
        k, v = line.strip().split("=", 1)
        env[k] = v
COMFY_DIR = env["COMFY_DIR"]
DRIVE_ROOT = env["DRIVE_ROOT"]
DRIVE_MODELS = env["DRIVE_MODELS"]

PORT = 8188
import os as _os
_os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
LOG = "/content/comfyui.log"
# A100 80GB / 40GB とも --highvram。足りない GPU だけ --lowvram
EXTRA = "--highvram"  #@param ["--highvram", "--normalvram", "--lowvram", "--novram"]

try:
    import torch
    if torch.cuda.is_available():
        _v = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"起動前 GPU: {torch.cuda.get_device_name(0)}  VRAM={_v:.1f}GB  EXTRA={EXTRA}")
        if _v < 50:
            print("※ 40GB のままです。14秒 R2V を通すならランタイムを A100 High Memory（80GB）にしてからセル1〜4をやり直してください。")
        else:
            print("※ 80GB クラス: セル8 は DURATION_S=14 / REF_IMAGE_SIZE=max の想定です。")
except Exception as _e:
    print("GPU probe skipped:", _e)

if not os.path.isfile(f"{COMFY_DIR}/main.py"):
    raise SystemExit("セル2が未実行です")

# リンクが切れていたら付け直す
def ensure_link(link_path, target_path):
    os.makedirs(target_path, exist_ok=True)
    if os.path.islink(link_path) and os.readlink(link_path) == target_path:
        return
    if os.path.islink(link_path) or os.path.exists(link_path):
        if os.path.islink(link_path):
            os.remove(link_path)
        elif os.path.isdir(link_path) and not os.path.islink(link_path):
            # 既に中身がある場合は触らない（Drive 接続済み想定）
            if os.listdir(link_path):
                return
            os.rmdir(link_path)
        else:
            os.remove(link_path)
    if not os.path.exists(link_path):
        os.symlink(target_path, link_path)

for sub in ["diffusion_models", "text_encoders", "vae", "loras", "clip_vision"]:
    ensure_link(f"{COMFY_DIR}/models/{sub}", f"{DRIVE_MODELS}/{sub}")
ensure_link(f"{COMFY_DIR}/output", f"{DRIVE_ROOT}/output")

# モデル存在チェック
need = [
    f"{DRIVE_MODELS}/diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors",
    f"{DRIVE_MODELS}/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
    f"{DRIVE_MODELS}/vae/minimax_h3_video_vae_fp16.safetensors",
    f"{DRIVE_MODELS}/vae/minimax_h3_audio_vae_fp32.safetensors",
]
# fp8 の場合も許容
alts = [
    f"{DRIVE_MODELS}/diffusion_models/minimax_h3_fl2va_pruned_fp8_scaled.safetensors",
    f"{DRIVE_MODELS}/diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors",
    f"{DRIVE_MODELS}/diffusion_models/minimax_h3_ref2va_pruned_fp8_scaled.safetensors",
]
ref2va_ok = any(
    os.path.exists(p) and os.path.getsize(p) > 1_000_000
    for p in [
        f"{DRIVE_MODELS}/diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors",
        f"{DRIVE_MODELS}/diffusion_models/minimax_h3_ref2va_pruned_fp8_scaled.safetensors",
    ]
)
if not ref2va_ok:
    print("⚠ ref2va が無いとセル8の R2V は動きません。セル3 MODE=both かセル5 を実行してください。")
missing = [p for p in need if not (os.path.exists(p) and os.path.getsize(p) > 1_000_000)]
if missing and not any(os.path.exists(a) and os.path.getsize(a) > 1_000_000 for a in alts):
    # fl2va だけ alt 可、TE/VAE は必須
    te_vae_miss = [p for p in need[1:] if not (os.path.exists(p) and os.path.getsize(p) > 1_000_000)]
    fl_ok = (os.path.exists(need[0]) and os.path.getsize(need[0]) > 1_000_000) or any(
        os.path.exists(a) and os.path.getsize(a) > 1_000_000 for a in alts
    )
    if te_vae_miss or not fl_ok:
        print("モデル不足。セル3を実行してください:")
        for p in missing:
            print(" -", p)
        raise SystemExit(1)

print("モデル確認 OK（Drive）")
!ls -lh "{DRIVE_MODELS}/diffusion_models" | head -10

for c in [
    f"fuser -k {PORT}/tcp",
    "pkill -f 'python.*main.py'",
    "pkill -f localtunnel",
    "pkill -f cloudflared",
]:
    subprocess.run(c, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)

os.chdir(COMFY_DIR)
log_f = open(LOG, "w", buffering=1)
cmd = [
    sys.executable, "main.py",
    "--listen", "0.0.0.0",
    "--port", str(PORT),
    EXTRA,
    "--disable-auto-launch",
    "--enable-cors-header",
]
print("starting:", " ".join(cmd))
proc = subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT, start_new_session=True)
print("PID:", proc.pid)

print("起動待ち...")
ok = False
for i in range(90):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=2)
        ok = True
        break
    except Exception:
        if proc.poll() is not None:
            break
        time.sleep(2)

if not ok:
    print(open(LOG, errors="replace").read()[-4000:])
    raise SystemExit("起動失敗")

print("✓ ComfyUI 起動OK")

# iframe
try:
    from google.colab import output as colab_output
    colab_output.serve_kernel_port_as_iframe(PORT, height=900)
    print("【方法A】下の埋め込み UI を使ってもOK")
except Exception as e:
    print("iframe:", e)

# localtunnel
if shutil.which("npm") is None:
    !curl -fsSL https://deb.nodesource.com/setup_20.x | bash - >/dev/null 2>&1
    !apt-get install -y -qq nodejs >/dev/null 2>&1

try:
    password = urllib.request.urlopen(
        "https://loca.lt/mytunnelpassword", timeout=15
    ).read().decode().strip()
except Exception:
    password = !curl -s https://loca.lt/mytunnelpassword
    password = password[0] if password else "?"

lt_log = "/content/localtunnel.log"
lt_f = open(lt_log, "w", buffering=1)
subprocess.Popen(
    ["npx", "--yes", "localtunnel", "--port", str(PORT)],
    stdout=lt_f, stderr=subprocess.STDOUT, start_new_session=True,
)

url = None
for _ in range(50):
    time.sleep(1)
    try:
        text = open(lt_log, errors="replace").read()
    except Exception:
        text = ""
    m = re.search(r"https://[a-z0-9.-]+\.loca\.lt", text, re.I)
    if m:
        url = m.group(0)
        break

print()
print("#" * 64)
print("#  開く URL はこれだけ（loca.lt）")
if url:
    print("# ", url)
    print("#  パスワード:", password)
else:
    print("#  自動取得失敗 → !npx --yes localtunnel --port 8188")
print("#  × colab.dev / trycloudflare は開かない")
print("#  生成結果 Drive:", f"{DRIVE_ROOT}/output")
print("#" * 64)

try:
    from IPython.display import display, HTML
    if url:
        display(HTML(f"""
        <div style="padding:16px;border:3px solid #0a0;background:#e8ffe8;font-size:18px">
          <b>ComfyUI を開く</b><br><br>
          <a href="{url}" target="_blank" style="font-size:22px">{url}</a><br><br>
          パスワード: <code style="font-size:20px">{password}</code><br><br>
          モデル・出力は Google Drive:<br>
          <code>{DRIVE_ROOT}</code><br><br>
          <span style="color:#a00">colab.dev は開かない</span>
        </div>
        """))
except Exception:
    pass

print("ログ:", LOG)
print("Drive output:", f"{DRIVE_ROOT}/output")


# ##############################################################################


In [ ]:
#@title セル5：（任意）R2V モデルを Drive に追加
# ##############################################################################
print("参照モード(R2V)が必要なときだけ")

import os
env = {}
with open("/content/h3_paths.env") as f:
    for line in f:
        k, v = line.strip().split("=", 1)
        env[k] = v
path = f"{env['DRIVE_MODELS']}/diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors"
url = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors"
if os.path.exists(path) and os.path.getsize(path) > 1_000_000:
    print("✓ すでに Drive にあります:", path)
else:
    print("↓ Drive へ DL:", path)
    !wget -c --show-progress -O "{path}" "{url}"
    print("完了")


# ##############################################################################


In [ ]:
#@title セル6：（任意）Drive 上のモデル一覧・容量
# ##############################################################################
import os
env = {}
with open("/content/h3_paths.env") as f:
    for line in f:
        k, v = line.strip().split("=", 1)
        env[k] = v
root = env["DRIVE_ROOT"]
print("Drive root:", root)
!du -sh "{root}" 2>/dev/null || true
!du -sh "{root}/models"/* 2>/dev/null || true
!find "{root}/models" -type f -printf "%s %p\n" 2>/dev/null | sort -n | tail -20


In [ ]:
#@title セル5b：LightX2V Turbo 4-step v1.0 LoRA を Drive に追加（高速化）
# ##############################################################################
print("=" * 60)
print(" セル5b：lightx2v turbo 4step v1.0 → models/loras")
print("=" * 60)

import os
from pathlib import Path

env = {}
with open("/content/h3_paths.env") as f:
    for line in f:
        k, v = line.strip().split("=", 1)
        env[k] = v

DRIVE_MODELS = env["DRIVE_MODELS"]
lora_dir = f"{DRIVE_MODELS}/loras"
os.makedirs(lora_dir, exist_ok=True)

LORA_NAME = "minimax_h3_fl2v_turbo_4step_v1.0_768p_comfyui_bf16.safetensors"
LORA_URL = "https://huggingface.co/lightx2v/Minimax-h3-Turbo/resolve/main/minimax_h3_fl2v_turbo_4step_v1.0_768p_comfyui_bf16.safetensors"
path = f"{lora_dir}/{LORA_NAME}"

if os.path.exists(path) and os.path.getsize(path) > 1_000_000:
    print("既にあります（スキップ）:")
    print(" ", path, f"({os.path.getsize(path)/1e6:.1f} MB)")
else:
    print("ダウンロード中…")
    print(LORA_URL)
    !wget -c --show-progress -O "{path}" "{LORA_URL}"
    if not (os.path.exists(path) and os.path.getsize(path) > 1_000_000):
        raise SystemExit("LoRA DL 失敗。URL とネットを確認してください")
    print("保存:", path, f"({os.path.getsize(path)/1e6:.1f} MB)")

comfy_lora = Path(env["COMFY_DIR"]) / "models" / "loras" / LORA_NAME
print("ComfyUI loras:", comfy_lora, "exists=", comfy_lora.exists())
print("次: セル8 で USE_LORA=True / STEPS=4。R2V は Ref2V turbo を使う（FL2V turbo ではない）")
!ls -lh "{lora_dir}" | tail -20



# Also fetch Ref2V turbo (required for R2V speed + quality; FL2V turbo is wrong model family)
REF2V_LORA = "minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors"
REF2V_URL = "https://huggingface.co/lightx2v/Minimax-h3-Turbo/resolve/main/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors"
_ref_path = f"{lora_dir}/{REF2V_LORA}"
if os.path.exists(_ref_path) and os.path.getsize(_ref_path) > 1_000_000:
    print("Ref2V turbo already present:", _ref_path)
else:
    print("Downloading Ref2V turbo...")
    import urllib.request
    urllib.request.urlretrieve(REF2V_URL, _ref_path)
    print("saved", _ref_path, os.path.getsize(_ref_path) if os.path.exists(_ref_path) else 0)


In [ ]:
#@title セル7：input 内の参照画像・動画を一覧（R2V 素材確認）
# ##############################################################################
print("=" * 60)
print(" セル7：参照素材一覧（Drive input）")
print("=" * 60)

import os
from pathlib import Path

env = {}
with open("/content/h3_paths.env") as f:
    for line in f:
        k, v = line.strip().split("=", 1)
        env[k] = v
DRIVE_ROOT = env["DRIVE_ROOT"]
COMFY_DIR = env["COMFY_DIR"]
INP = Path(COMFY_DIR) / "input"
os.makedirs(INP, exist_ok=True)

IMG_EXT = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
VID_EXT = {".mp4", ".mov", ".webm", ".mkv", ".avi"}

images = sorted(
    [p for p in INP.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXT],
    key=lambda p: p.name.lower(),
)
videos = sorted(
    [p for p in INP.rglob("*") if p.is_file() and p.suffix.lower() in VID_EXT],
    key=lambda p: p.name.lower(),
)

print(f"input dir: {INP}")
print(f"images: {len(images)}  (R2V max 9)")
for i, p in enumerate(images, 1):
    print(f"  [{i:02d}] {p.relative_to(INP)}  ({p.stat().st_size/1e6:.2f} MB)")
print(f"videos: {len(videos)}  (R2V motion max 3)")
for i, p in enumerate(videos, 1):
    print(f"  [{i:02d}] {p.relative_to(INP)}  ({p.stat().st_size/1e6:.2f} MB)")

if not images and not videos:
    print("\n素材なし。Drive に置いてください:")
    print(f"  {DRIVE_ROOT}/input/")
else:
    print("\n次は【セル8】で IMAGE_FILES / VIDEO_FILES を指定して R2V")



In [ ]:
%%writefile h3_r2v_core.py
"""Helpers for MiniMax H3 ComfyUI R2V (identity from stills, motion from video).

No ComfyUI / network required. Used by minimax_h3_colab_完全版.ipynb cell 8.
"""

from __future__ import annotations

from pathlib import Path
from typing import Any


def frames(duration_s: float) -> int:
    """H3 length grid: 17k+5 at 24fps."""
    base = max(5, int(round(float(duration_s) * 24)))
    return int(base + (5 - (base % 17)) % 17)


def parse_list(s: str) -> list[str]:
    s = (s or "").strip()
    if not s or s.lower() in ("none", "-", "null"):
        return []
    return [x.strip().lstrip("./") for x in s.replace(";", ",").split(",") if x.strip()]


def comfy_media_name(rel: str | Path) -> str:
    """ComfyUI LoadImage / VHS video widgets want a path relative to input/, not basename-only."""
    return str(rel).replace("\\", "/").lstrip("./")


def prefer_ref2v_lora(lora_paths: list[Path], use_lora: bool) -> str | None:
    """Prefer Ref2V turbo for ref2va unet; never prefer FL2V-only when ref2v exists."""
    if not use_lora or not lora_paths:
        return None
    names = [p.name for p in lora_paths if p.suffix.lower() == ".safetensors"]
    if not names:
        return None

    def score(n: str) -> tuple:
        nl = n.lower()
        is_ref2v = 1 if ("ref2v" in nl or "ref2va" in nl) else 0
        is_fl2v = 1 if ("fl2v" in nl or "fl2va" in nl) and not is_ref2v else 0
        is_turbo = 1 if "turbo" in nl else 0
        is_4step = 1 if "4step" in nl or "4_step" in nl else 0
        is_comfy = 1 if "comfyui" in nl else 0
        return (is_ref2v, is_turbo, is_4step, is_comfy, -is_fl2v, n)

    ref2v = [n for n in names if "ref2v" in n.lower() or "ref2va" in n.lower()]
    if ref2v:
        return sorted(ref2v, key=score, reverse=True)[0]
    return sorted(names, key=score, reverse=True)[0]


def role_lock_preamble(img_names: list[str], vid_names: list[str]) -> str:
    lines = [
        "ROLE LOCK (mandatory):",
        "REFERENCE VIDEO = MOTION ONLY (hard rule):",
        "- From every <Video N>, read ONLY: body motion, hand trajectories, footwork, camera path,",
        "  framing changes, pacing, cut rhythm, and timing.",
        "- From every <Video N>, IGNORE completely: faces, gender presentation, age, hair, skin tone,",
        "  body build, costumes, logos, on-screen text, and the original actor's identity.",
        "- Never retarget appearance from the motion clip. The motion clip is a choreography/camera guide only.",
    ]
    for i, n in enumerate(img_names):
        lines.append(
            f"- <Picture {i+1}> ({n}) = IDENTITY + COSTUME only. "
            "Copy exact face, hair, skin, body proportions, and wardrobe from this still. "
            "Do NOT replace this person with anyone visible in any motion video."
        )
    for i, n in enumerate(vid_names):
        lines.append(
            f"- <Video {i+1}> ({n}) = MOTION + CAMERA + TIMING ONLY (not appearance). "
            "Follow camera path, pacing, body action, shot rhythm. "
            "Do NOT invent different choreography. "
            "Do NOT copy faces, body type, age, hair, or costumes from people in this video. "
            "Drive the still-locked characters through this motion like motion capture."
        )
    if not vid_names:
        lines.append("- No motion video: invent plausible cinematic motion consistent with the stills.")
    lines.append(
        "CONFLICT RULE: appearance always wins from Pictures; motion always wins from Videos. "
        "If the video actors look different from the stills, keep the still faces and only transfer motion."
    )
    return "\n".join(lines) + "\n\n"


def build_default_prompt(img_names: list[str], vid_names: list[str], duration_s: float) -> str:
    lock = role_lock_preamble(img_names, vid_names)
    pic_lines = [f"- <Picture {i+1}> = {n}" for i, n in enumerate(img_names)]
    vid_lines = [
        f"- <Video {i+1}> = {n} (MOTION ONLY: body/camera/timing — ignore faces & costumes in this clip)"
        for i, n in enumerate(vid_names)
    ]
    extra = ""
    if vid_names:
        extra = (
            " Transfer ONLY motion and camera from <Video 1> onto those still-locked characters "
            "(motion-capture style). Do not inherit the video actors' appearance."
        )
    pics = " and <Picture 2>" if len(img_names) > 1 else ""
    return (
        lock
        + "Use these references:\n"
        + "\n".join(pic_lines + vid_lines)
        + "\nSTYLE: photorealistic live-action cinematic, real skin pores, natural materials, "
        "no anime cel, no text, no subtitles, no logos.\n"
        f"integrated_multimodal_description: [Shot 1] Live-action remake for about "
        f"{duration_s:.0f} seconds. Characters must match <Picture 1>{pics} faces exactly."
        + extra
        + "\noverall_soundscape: Ambient and action SFX matching motion."
        + "\nnon_diegetic_music: None."
    )


def finalize_prompt(
    prompt: str,
    img_names: list[str],
    vid_names: list[str],
    duration_s: float,
    inject_role_lock: bool = True,
) -> str:
    raw = (prompt or "").strip()
    if not raw:
        return build_default_prompt(img_names, vid_names, duration_s)
    pics = "\n".join(f"<Picture {i+1}>:{n}" for i, n in enumerate(img_names))
    vids = "\n".join(f"<Video {i+1}>:{n}" for i, n in enumerate(vid_names))
    out = raw.replace("{pictures}", pics).replace("{videos}", vids)
    if inject_role_lock:
        has_lock = "ROLE LOCK" in out or "MOTION LOCK" in out or "APPEARANCE LOCK" in out
        if not has_lock:
            out = role_lock_preamble(img_names, vid_names) + out
        if img_names and "<Picture 1>" not in out and "<picture 1>" not in out.lower():
            out = f"Identity for character 1 is locked to <Picture 1> ({img_names[0]}).\n" + out
        if len(img_names) > 1 and "<Picture 2>" not in out:
            out = f"Identity for character 2 is locked to <Picture 2> ({img_names[1]}).\n" + out
        if vid_names and "<Video 1>" not in out:
            out = (
                f"MOTION ONLY from <Video 1> ({vid_names[0]}): body action, camera, timing. "
                "Ignore faces/costumes in the video; keep still-image identity.\n"
                + out
            )
        if vid_names and "MOTION ONLY" not in out.upper() and "motion only" not in out.lower():
            out = (
                "HARD CONSTRAINT: Reference videos provide MOTION ONLY "
                "(choreography + camera + timing). Appearance comes exclusively from still Pictures.\n"
                + out
            )
    return out


def image_ref_key(i: int) -> str:
    return f"ref_images.ref_image_{i}"


def video_ref_key(i: int) -> str:
    return f"ref_videos.ref_video_{i}"


def gpu_vram_tier(vram_gb: float) -> str:
    """40GB A100 vs 80GB A100 High Memory. Host RAM is ignored."""
    v = float(vram_gb)
    if v >= 70:
        return "80plus"
    if v >= 32:
        return "40"
    if v >= 20:
        return "24"
    return "low"


def format_gpu_runtime_note(
    vram_gb: float,
    *,
    host_ram_gb: float | None = None,
    gpu_name: str = "",
) -> str:
    """Tell High-RAM (CPU) apart from A100 High Memory (80GB VRAM)."""
    name = gpu_name or "GPU"
    ram = f"  host RAM={host_ram_gb:.0f}GB" if host_ram_gb else ""
    head = f"{name}  VRAM={float(vram_gb):.1f}GB{ram}"
    tier = gpu_vram_tier(vram_gb)
    if tier == "80plus":
        return head + "\n判定: 80GB クラス。14秒 + REF_IMAGE_SIZE=max + 参照動画をそのまま通します。"
    if tier == "40":
        extra = ""
        if host_ram_gb is not None and host_ram_gb >= 50:
            extra = (
                " システムRAMは足りていますが、前回の OOM は GPU VRAM 不足です。"
                " Colab の High-RAM は CPU メモリなので、これでは 14秒 R2V は通りません。"
                " ランタイムで A100 High Memory（80GB GPU）を選んでください。"
            )
        return (
            head
            + "\n判定: 40GB クラス。14秒 + max + 参照動画は OOM するので、先に約6秒で回します。"
            + extra
        )
    if tier == "24":
        return head + "\n判定: 24GB クラス。R2V は 5秒・match 寄りになります。"
    return head + "\n判定: VRAM 不足。GPU ランタイムを選んでください。"


def cap_duration_for_vram(
    duration_s: float,
    *,
    vram_gb: float,
    n_images: int,
    has_video: bool,
    ref_image_size: str,
) -> float:
    """R2V VRAM scales with frames × ref_image_size × video tokens. 14s+max OOMs on 40GB."""
    duration_s = float(duration_s)
    if duration_s < 1:
        duration_s = 5
    if not has_video:
        return min(duration_s, 15.0)
    tier = gpu_vram_tier(vram_gb)
    if tier == "low":
        cap = 5.0
    elif tier == "24":
        cap = 5.0
    elif tier == "40":
        # A100 40GB: 14s + max + 2 stills + motion clip requested 18.5GiB extra and died
        cap = 6.0 if (ref_image_size == "max" and n_images >= 2) else 8.0
    else:
        # A100 80GB High Memory: peak request was ~18.5GiB extra; keep full length
        cap = 15.0
    return min(duration_s, cap)


def r2v_retry_plans(
    *,
    duration_s: float,
    ref_image_size: str,
    width: int,
    height: int,
    n_images: int,
    has_video: bool,
    vram_gb: float,
) -> list[dict[str, Any]]:
    """Smaller later. Never drops the motion video."""
    size = ref_image_size if ref_image_size in ("match", "max") else "max"
    first_dur = cap_duration_for_vram(
        duration_s,
        vram_gb=vram_gb,
        n_images=n_images,
        has_video=has_video,
        ref_image_size=ref_image_size,
    )
    tier = gpu_vram_tier(vram_gb)
    if has_video and tier == "80plus":
        plans = [
            {
                "duration_s": first_dur,
                "ref_image_size": size,
                "width": width,
                "height": height,
                "motion_max_edge": None,
                "label": f"80GB dur={first_dur:.0f}s size={size} motion=native",
            },
            {
                "duration_s": min(first_dur, 10.0),
                "ref_image_size": size,
                "width": width,
                "height": height,
                "motion_max_edge": 768,
                "label": "80GB fallback dur<=10s motion_edge=768",
            },
            {
                "duration_s": 6.0,
                "ref_image_size": "match",
                "width": width,
                "height": height,
                "motion_max_edge": 640,
                "label": "dur=6s size=match motion_edge=640",
            },
        ]
    else:
        plans = [
            {
                "duration_s": first_dur,
                "ref_image_size": size,
                "width": width,
                "height": height,
                "motion_max_edge": 768 if has_video else None,
                "label": f"dur={first_dur:.0f}s size={size} motion_edge=768",
            }
        ]
        if has_video:
            plans.append(
                {
                    "duration_s": min(first_dur, 6.0),
                    "ref_image_size": "match",
                    "width": width,
                    "height": height,
                    "motion_max_edge": 640,
                    "label": "dur<=6s size=match motion_edge=640",
                }
            )
            plans.append(
                {
                    "duration_s": 5.0,
                    "ref_image_size": "match",
                    "width": min(width, 768) if width >= height else width,
                    "height": min(height, 448) if width >= height else min(height, 768),
                    "motion_max_edge": 512,
                    "label": "dur=5s size=match 768-class canvas motion_edge=512",
                }
            )
    # snap spatial to multiple of 32
    for p in plans:
        p["width"] = max(32, int(p["width"]) // 32 * 32)
        p["height"] = max(32, int(p["height"]) // 32 * 32)
    # de-dupe identical plans
    out: list[dict[str, Any]] = []
    seen: set[tuple] = set()
    for p in plans:
        key = (p["duration_s"], p["ref_image_size"], p["width"], p["height"], p["motion_max_edge"])
        if key in seen:
            continue
        seen.add(key)
        out.append(p)
    return out


def is_oom_error(payload: Any) -> bool:
    text = str(payload).lower()
    return "out of memory" in text or "outofmemory" in text or "cuda oom" in text


def vhs_load_video_inputs(
    object_info: dict[str, Any] | None,
    filename: str,
    length: int,
    motion_max_edge: int | None = 768,
) -> dict[str, Any]:
    """Fill VHS_LoadVideo widgets from live object_info so schema drift does not drop the clip."""
    filename = comfy_media_name(filename)
    info = ((object_info or {}).get("VHS_LoadVideo") or {}).get("input") or {}
    required = info.get("required") or {}
    optional = info.get("optional") or {}
    merged = {**required, **optional}
    inputs: dict[str, Any] = {}
    for name, spec in merged.items():
        if isinstance(spec, list) and len(spec) > 1 and isinstance(spec[1], dict) and "default" in spec[1]:
            inputs[name] = spec[1]["default"]
    if "video" in merged:
        inputs["video"] = filename
    elif "file" in merged:
        inputs["file"] = filename
    else:
        inputs["video"] = filename
    if "force_rate" in merged:
        inputs["force_rate"] = 24
    if "frame_load_cap" in merged:
        inputs["frame_load_cap"] = int(length)
    if "skip_first_frames" in merged:
        inputs["skip_first_frames"] = 0
    if "select_every_nth" in merged:
        inputs["select_every_nth"] = 1
    if motion_max_edge:
        _apply_vhs_motion_size(inputs, merged, int(motion_max_edge))
    elif "force_size" in merged:
        inputs["force_size"] = "Disabled"
    return inputs


def _apply_vhs_motion_size(inputs: dict[str, Any], merged: dict[str, Any], max_edge: int) -> None:
    """Downscale motion frames. Full-res 14s clips blow 40GB during sampling."""
    max_edge = max(256, int(max_edge))
    if "force_size" in merged:
        spec = merged["force_size"]
        choices = spec[0] if isinstance(spec, list) and spec else []
        picked = None
        if isinstance(choices, list):
            for cand in ("Custom", "Custom Width", str(max_edge), "768", "512"):
                if cand in choices:
                    picked = cand
                    break
        inputs["force_size"] = picked or "Custom"
    if "custom_width" in merged:
        inputs["custom_width"] = max_edge
    if "custom_height" in merged:
        h = max(256, (max_edge * 9 // 16) // 2 * 2)
        inputs["custom_height"] = h


def native_load_video_inputs(filename: str) -> dict[str, Any]:
    name = comfy_media_name(filename)
    return {"file": name, "video": name}


def build_r2v_graph(
    *,
    img_names: list[str],
    vid_names: list[str],
    prompt: str,
    unet: str,
    lora_name: str | None,
    lora_strength: float,
    width: int,
    height: int,
    duration_s: float,
    seed: int,
    steps: int,
    filename_prefix: str,
    ref_image_size: str = "max",
    use_videos: bool = True,
    has_vhs: bool = True,
    has_lora_loader: bool = True,
    has_audio_decode: bool = True,
    object_info: dict[str, Any] | None = None,
    motion_max_edge: int | None = 768,
    clip_name: str = "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
    vvae: str = "minimax_h3_video_vae_fp16.safetensors",
    avae: str = "minimax_h3_audio_vae_fp32.safetensors",
) -> dict[str, Any]:
    """Build ComfyUI API graph for MiniMaxH3ReferenceToVideo."""
    if width % 32 or height % 32:
        raise ValueError(f"H3 width/height must be multiples of 32, got {width}x{height}")
    g: dict[str, Any] = {}
    length = frames(duration_s)

    for i, fname in enumerate(img_names):
        g[str(100 + i)] = {
            "class_type": "LoadImage",
            "inputs": {"image": comfy_media_name(fname)},
        }

    g["1"] = {
        "class_type": "UNETLoader",
        "inputs": {"unet_name": unet, "weight_dtype": "default"},
    }
    model: list[Any] = ["1", 0]
    if lora_name and has_lora_loader:
        g["2"] = {
            "class_type": "LoraLoaderModelOnly",
            "inputs": {
                "model": ["1", 0],
                "lora_name": lora_name,
                "strength_model": float(lora_strength),
            },
        }
        model = ["2", 0]

    g["3"] = {
        "class_type": "CLIPLoader",
        "inputs": {"clip_name": clip_name, "type": "minimax", "device": "default"},
    }
    g["4"] = {"class_type": "VAELoader", "inputs": {"vae_name": vvae}}
    g["5"] = {"class_type": "VAELoader", "inputs": {"vae_name": avae}}

    r_inputs: dict[str, Any] = {
        "clip": ["3", 0],
        "vae": ["4", 0],
        "audio_vae": ["5", 0],
        "prompt": prompt,
        "width": int(width),
        "height": int(height),
        "length": length,
        "ref_image_size": ref_image_size if ref_image_size in ("match", "max") else "max",
    }

    for i in range(len(img_names)):
        r_inputs[image_ref_key(i)] = [str(100 + i), 0]

    if use_videos and vid_names:
        for vi, vname in enumerate(vid_names[:3]):
            node_id = str(190 + vi)
            if has_vhs:
                g[node_id] = {
                    "class_type": "VHS_LoadVideo",
                    "inputs": vhs_load_video_inputs(
                        object_info, vname, length, motion_max_edge=motion_max_edge
                    ),
                }
            else:
                g[node_id] = {
                    "class_type": "LoadVideo",
                    "inputs": native_load_video_inputs(vname),
                }
            r_inputs[video_ref_key(vi)] = [node_id, 0]

    for bad in ("ref_videos", "ref_images", "ref_audios", "ref_video_audios"):
        r_inputs.pop(bad, None)

    g["20"] = {"class_type": "MiniMaxH3ReferenceToVideo", "inputs": r_inputs}
    g["21"] = {"class_type": "RandomNoise", "inputs": {"noise_seed": int(seed)}}
    sampler = "euler" if lora_name else "res_multistep"
    scheduler = "simple" if lora_name else "beta"
    g["22"] = {"class_type": "KSamplerSelect", "inputs": {"sampler_name": sampler}}
    g["23"] = {
        "class_type": "BasicScheduler",
        "inputs": {
            "model": model,
            "scheduler": scheduler,
            "steps": int(steps) if lora_name else max(int(steps), 16),
            "denoise": 1.0,
        },
    }
    g["24"] = {
        "class_type": "BasicGuider",
        "inputs": {"model": model, "conditioning": ["20", 0]},
    }
    g["25"] = {
        "class_type": "SamplerCustomAdvanced",
        "inputs": {
            "noise": ["21", 0],
            "guider": ["24", 0],
            "sampler": ["22", 0],
            "sigmas": ["23", 0],
            "latent_image": ["20", 1],
        },
    }
    g["26"] = {
        "class_type": "VAEDecode",
        "inputs": {"samples": ["25", 0], "vae": ["4", 0]},
    }
    if has_audio_decode:
        g["27"] = {
            "class_type": "VAEDecodeAudio",
            "inputs": {"samples": ["25", 0], "vae": ["5", 0]},
        }
        g["28"] = {
            "class_type": "CreateVideo",
            "inputs": {"images": ["26", 0], "audio": ["27", 0], "fps": 24},
        }
    else:
        g["28"] = {
            "class_type": "CreateVideo",
            "inputs": {"images": ["26", 0], "fps": 24},
        }
    g["29"] = {
        "class_type": "SaveVideo",
        "inputs": {
            "video": ["28", 0],
            "filename_prefix": filename_prefix,
            "format": "auto",
            "codec": "auto",
        },
    }
    return g


def assert_graph_identity_motion(
    graph: dict[str, Any],
    *,
    expect_images: int,
    expect_videos: int,
    prompt: str,
) -> list[str]:
    """Return list of error strings (empty = ok)."""
    errs: list[str] = []
    node = graph.get("20") or {}
    if node.get("class_type") != "MiniMaxH3ReferenceToVideo":
        errs.append("node 20 is not MiniMaxH3ReferenceToVideo")
        return errs
    inputs = node.get("inputs") or {}
    for bad in ("ref_videos", "ref_images"):
        if bad in inputs:
            errs.append(f"parent key {bad} must not be used alone")
    for i in range(expect_images):
        k = image_ref_key(i)
        if k not in inputs:
            errs.append(f"missing {k}")
    if expect_videos:
        for i in range(expect_videos):
            k = video_ref_key(i)
            if k not in inputs:
                errs.append(f"missing {k}")
        if "190" not in graph:
            errs.append("missing video load node 190")
        else:
            vin = graph["190"].get("inputs") or {}
            klass = graph["190"].get("class_type")
            if klass == "VHS_LoadVideo" and int(vin.get("force_rate") or 0) not in (0, 24):
                # 0 = keep source rate in some VHS versions; 24 is required by H3
                errs.append("VHS force_rate must be 24")
            media = vin.get("video") or vin.get("file") or ""
            if not media:
                errs.append("video loader has empty filename")
    ris = inputs.get("ref_image_size")
    if ris not in ("match", "max"):
        errs.append(f"bad ref_image_size: {ris}")
    if expect_images and "<Picture 1>" not in prompt and "Picture 1" not in prompt:
        errs.append("prompt missing Picture 1 identity lock")
    if expect_videos and "<Video 1>" not in prompt and "Video 1" not in prompt:
        errs.append("prompt missing Video 1 motion lock")
    if expect_videos:
        pu = prompt.upper()
        if "MOTION ONLY" not in pu and "MOTION + CAMERA" not in pu and "MOTION/CAMERA" not in pu:
            if "MOTION" not in pu:
                errs.append("prompt missing MOTION-only language for reference video")
        if "FACE" not in pu and "IDENTITY" not in pu and "PICTURE" not in pu:
            errs.append("prompt missing still-identity vs video-motion separation")
    return errs


def _find_ffmpeg() -> str | None:
    import shutil

    exe = shutil.which("ffmpeg")
    if exe:
        return exe
    for c in ("/usr/bin/ffmpeg", "/usr/local/bin/ffmpeg"):
        if Path(c).is_file():
            return c
    return None


def strip_motion_identity_video(
    src: Path,
    dst: Path,
    *,
    mode: str = "edges",
    ffmpeg_bin: str | None = None,
) -> Path:
    """Optional last resort: destroy photoreal identity while keeping coarse motion."""
    import subprocess

    src = Path(src)
    dst = Path(dst)
    if not src.is_file():
        raise FileNotFoundError(src)
    dst.parent.mkdir(parents=True, exist_ok=True)
    mode = (mode or "edges").lower().strip()
    if mode not in ("edges", "blur"):
        mode = "edges"
    ff = ffmpeg_bin or _find_ffmpeg()
    if not ff:
        raise RuntimeError("Cannot strip video identity: ffmpeg is required")
    if mode == "blur":
        vf = "format=gray,gblur=sigma=12,format=yuv420p"
    else:
        vf = "format=gray,edgedetect=mode=colormix:high=0.12:low=0.04,format=yuv420p"
    r = subprocess.run(
        [ff, "-y", "-i", str(src), "-vf", vf, "-an", "-c:v", "libx264", "-pix_fmt", "yuv420p", str(dst)],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0 or not dst.is_file() or dst.stat().st_size < 1000:
        raise RuntimeError(f"ffmpeg strip failed: {(r.stderr or '')[-800:]}")
    return dst


def prepare_motion_refs(
    inp_dir: Path,
    rel_names: list[str],
    *,
    enabled: bool = True,
    mode: str = "edges",
) -> list[str]:
    if not enabled or not rel_names:
        return list(rel_names)
    out_names: list[str] = []
    for rel in rel_names:
        src = Path(inp_dir) / rel
        if not src.is_file():
            hits = list(Path(inp_dir).rglob(Path(rel).name))
            if not hits:
                raise FileNotFoundError(rel)
            src = hits[0]
        stem = src.stem
        if stem.endswith("_motion_only_edges") or stem.endswith("_motion_only_blur"):
            try:
                out_names.append(str(src.relative_to(inp_dir)).replace("\\", "/"))
            except ValueError:
                out_names.append(src.name)
            continue
        suffix = "_motion_only_edges" if mode == "edges" else "_motion_only_blur"
        dst = src.with_name(f"{stem}{suffix}.mp4")
        strip_motion_identity_video(src, dst, mode=mode)
        try:
            out_names.append(str(dst.relative_to(inp_dir)).replace("\\", "/"))
        except ValueError:
            out_names.append(dst.name)
    return out_names


In [ ]:
%%writefile pose_motion_lock.py
"""Pose-locked character replacement for action footage.

MiniMax H3 R2V is NOT motion capture: it re-generates from a video hint.
Action (fights, swords, fast camera) needs per-frame pose + Mix replace.

This module builds a ComfyUI API graph for Wan 2.2 Animate:
  still image = identity + costume
  DWPose video = body/hand motion
  Mix mode = drop the new person into the original clip (camera/timing stay)
"""

from __future__ import annotations

from pathlib import Path
from typing import Any

from h3_r2v_core import comfy_media_name, vhs_load_video_inputs

H3_NOT_MOCAP = (
    "MiniMax H3 R2V はモーションキャプチャではない。"
    "参照動画を見て再生成するだけなので、激しいアクションは手足・刀・カメラがずれる。"
    "完全模倣したい場合は Wan 2.2 Animate Mix（骨格ロック＋元映像へ置換）を使う。"
)

WAN_MODELS = {
    "unet": (
        "Wan2_2-Animate-14B_fp8_e4m3fn_scaled_KJ.safetensors",
        "https://huggingface.co/Kijai/WanVideo_comfy_fp8_scaled/resolve/main/Wan22Animate/Wan2_2-Animate-14B_fp8_e4m3fn_scaled_KJ.safetensors",
        "diffusion_models",
    ),
    "lora": (
        "lightx2v_I2V_14B_480p_cfg_step_distill_rank64_bf16.safetensors",
        "https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/Lightx2v/lightx2v_I2V_14B_480p_cfg_step_distill_rank64_bf16.safetensors",
        "loras",
    ),
    "relight": (
        "WanAnimate_relight_lora_fp16.safetensors",
        "https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/LoRAs/Wan22_relight/WanAnimate_relight_lora_fp16.safetensors",
        "loras",
    ),
    "clip": (
        "umt5_xxl_fp8_e4m3fn_scaled.safetensors",
        "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
        "text_encoders",
    ),
    "clip_vision": (
        "clip_vision_h.safetensors",
        "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/clip_vision/clip_vision_h.safetensors",
        "clip_vision",
    ),
    "vae": (
        "wan_2.1_vae.safetensors",
        "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors",
        "vae",
    ),
}

CUSTOM_NODES = {
    "comfyui_controlnet_aux": "https://github.com/Fannovel16/comfyui_controlnet_aux.git",
}


def wan_length(duration_s: float, fps: float = 16.0, chunk: int = 77) -> int:
    """Wan Animate length is 4k+1. Official Mix chunk is 77 (~4.8s @16fps)."""
    n = max(5, int(round(float(duration_s) * float(fps))))
    n = n + (1 - n % 4) % 4
    if n % 4 != 1:
        n += 1
    return min(int(n), int(chunk))


def chunk_count(duration_s: float, fps: float = 16.0, chunk: int = 77) -> int:
    total = max(1, int(round(float(duration_s) * float(fps))))
    return max(1, (total + chunk - 1) // chunk)


def assign_people_left_to_right(
    boxes: list[tuple[Any, float, float, float, float]],
) -> dict[Any, int]:
    """Map detector id → person index 0..n-1 by x-center (left = Image 1)."""
    ranked = sorted(boxes, key=lambda b: (b[1] + b[3]) / 2.0)
    return {b[0]: i for i, b in enumerate(ranked)}


def mix_pass_plan(img_names: list[str], video_name: str) -> list[dict[str, Any]]:
    """Two-person fight: replace left person, then right person, keeping camera."""
    jobs: list[dict[str, Any]] = []
    background = video_name
    for i, img in enumerate(img_names):
        out_prefix = f"video/h3_mix_pass{i + 1}"
        jobs.append(
            {
                "pass_index": i,
                "reference_image": img,
                "background_video": background,
                "person_index": i,
                "filename_prefix": out_prefix,
                "label": f"mix pass{i + 1}: {img} into {background}",
            }
        )
        background = out_prefix
    return jobs


def dwpose_inputs(*, body: bool, face: bool, hands: bool, resolution: int = 512) -> dict[str, Any]:
    on, off = "enable", "disable"
    return {
        "detect_hand": on if hands else off,
        "detect_body": on if body else off,
        "detect_face": on if face else off,
        "resolution": int(resolution),
        "bbox_detector": "yolox_l.onnx",
        "pose_estimator": "dw-ll_ucoco_384_bs5.torchscript.pt",
    }


def build_pose_preview_graph(
    *,
    video_name: str,
    filename_prefix: str = "video/pose_preview",
    length: int = 77,
    fps: float = 16.0,
    object_info: dict[str, Any] | None = None,
) -> dict[str, Any]:
    """DWPose only. Inspect this before spending a Mix run on a fight clip."""
    g: dict[str, Any] = {}
    g["190"] = {
        "class_type": "VHS_LoadVideo",
        "inputs": vhs_load_video_inputs(object_info, video_name, length, motion_max_edge=None),
    }
    g["101"] = {
        "class_type": "DWPreprocessor",
        "inputs": {"image": ["190", 0], **dwpose_inputs(body=True, face=False, hands=True)},
    }
    g["29"] = {
        "class_type": "CreateVideo",
        "inputs": {"images": ["101", 0], "fps": float(fps)},
    }
    g["30"] = {
        "class_type": "SaveVideo",
        "inputs": {
            "video": ["29", 0],
            "filename_prefix": filename_prefix,
            "format": "auto",
            "codec": "auto",
        },
    }
    return g


def build_wan_animate_graph(
    *,
    image_name: str,
    video_name: str,
    prompt: str,
    negative: str = "text, watermark, logo, subtitles, blurry, extra limbs",
    mode: str = "mix",
    mask_name: str | None = None,
    unet: str = WAN_MODELS["unet"][0],
    lora_name: str | None = WAN_MODELS["lora"][0],
    relight_lora: str | None = WAN_MODELS["relight"][0],
    clip_name: str = WAN_MODELS["clip"][0],
    clip_vision_name: str = WAN_MODELS["clip_vision"][0],
    vae_name: str = WAN_MODELS["vae"][0],
    width: int = 640,
    height: int = 368,
    length: int = 77,
    fps: float = 16.0,
    seed: int = 42,
    steps: int = 4,
    cfg: float = 1.0,
    filename_prefix: str = "video/wan_animate_mix",
    object_info: dict[str, Any] | None = None,
    grow_mask: int = 28,
) -> dict[str, Any]:
    """One Mix/Move pass. Mix keeps the source camera; Move re-stages onto the still."""
    if width % 16 or height % 16:
        raise ValueError(f"Wan Animate width/height must be multiples of 16, got {width}x{height}")
    mode = (mode or "mix").lower()
    if mode not in ("mix", "move"):
        raise ValueError("mode must be mix or move")
    if mode == "mix" and not mask_name:
        raise ValueError("mix mode needs a per-person mask video (character_mask)")

    g: dict[str, Any] = {}
    g["10"] = {
        "class_type": "LoadImage",
        "inputs": {"image": comfy_media_name(image_name)},
    }
    g["190"] = {
        "class_type": "VHS_LoadVideo",
        "inputs": vhs_load_video_inputs(object_info, video_name, length, motion_max_edge=None),
    }
    g["101"] = {
        "class_type": "DWPreprocessor",
        "inputs": {"image": ["190", 0], **dwpose_inputs(body=True, face=False, hands=True)},
    }
    g["100"] = {
        "class_type": "DWPreprocessor",
        "inputs": {"image": ["190", 0], **dwpose_inputs(body=False, face=True, hands=False)},
    }
    g["14"] = {
        "class_type": "CLIPLoader",
        "inputs": {"clip_name": clip_name, "type": "wan", "device": "default"},
    }
    g["15"] = {
        "class_type": "CLIPTextEncode",
        "inputs": {"clip": ["14", 0], "text": prompt},
    }
    g["16"] = {
        "class_type": "CLIPTextEncode",
        "inputs": {"clip": ["14", 0], "text": negative},
    }
    g["17"] = {
        "class_type": "UNETLoader",
        "inputs": {"unet_name": unet, "weight_dtype": "default"},
    }
    model: list[Any] = ["17", 0]
    next_id = 18
    if lora_name:
        g[str(next_id)] = {
            "class_type": "LoraLoaderModelOnly",
            "inputs": {"model": model, "lora_name": lora_name, "strength_model": 1.0},
        }
        model = [str(next_id), 0]
        next_id += 1
    if mode == "mix" and relight_lora:
        g[str(next_id)] = {
            "class_type": "LoraLoaderModelOnly",
            "inputs": {"model": model, "lora_name": relight_lora, "strength_model": 1.0},
        }
        model = [str(next_id), 0]
        next_id += 1
    g["20"] = {
        "class_type": "ModelSamplingSD3",
        "inputs": {"model": model, "shift": 8.0},
    }
    g["21"] = {"class_type": "VAELoader", "inputs": {"vae_name": vae_name}}
    g["22"] = {
        "class_type": "CLIPVisionLoader",
        "inputs": {"clip_name": clip_vision_name},
    }
    g["23"] = {
        "class_type": "CLIPVisionEncode",
        "inputs": {"clip_vision": ["22", 0], "image": ["10", 0], "crop": "center"},
    }

    wan_in: dict[str, Any] = {
        "positive": ["15", 0],
        "negative": ["16", 0],
        "vae": ["21", 0],
        "clip_vision_output": ["23", 0],
        "reference_image": ["10", 0],
        "face_video": ["100", 0],
        "pose_video": ["101", 0],
        "width": int(width),
        "height": int(height),
        "length": int(length),
        "batch_size": 1,
        "continue_motion_max_frames": 5,
        "video_frame_offset": 0,
    }
    if mode == "mix":
        g["191"] = {
            "class_type": "VHS_LoadVideo",
            "inputs": vhs_load_video_inputs(object_info, mask_name, length, motion_max_edge=None),
        }
        g["24"] = {
            "class_type": "ImageToMask",
            "inputs": {"image": ["191", 0], "channel": "red"},
        }
        g["241"] = {
            "class_type": "GrowMask",
            "inputs": {"mask": ["24", 0], "expand": int(grow_mask), "tapered_corners": True},
        }
        wan_in["background_video"] = ["190", 0]
        wan_in["character_mask"] = ["241", 0]

    g["25"] = {"class_type": "WanAnimateToVideo", "inputs": wan_in}
    g["26"] = {
        "class_type": "KSampler",
        "inputs": {
            "model": ["20", 0],
            "positive": ["25", 0],
            "negative": ["25", 1],
            "latent_image": ["25", 2],
            "seed": int(seed),
            "steps": int(steps),
            "cfg": float(cfg),
            "sampler_name": "euler",
            "scheduler": "simple",
            "denoise": 1.0,
        },
    }
    g["27"] = {
        "class_type": "TrimVideoLatent",
        "inputs": {"samples": ["26", 0], "trim_amount": ["25", 3]},
    }
    g["28"] = {
        "class_type": "VAEDecode",
        "inputs": {"samples": ["27", 0], "vae": ["21", 0]},
    }
    g["29"] = {
        "class_type": "CreateVideo",
        "inputs": {"images": ["28", 0], "fps": float(fps)},
    }
    g["30"] = {
        "class_type": "SaveVideo",
        "inputs": {
            "video": ["29", 0],
            "filename_prefix": filename_prefix,
            "format": "auto",
            "codec": "auto",
        },
    }
    return g


def assert_graph_pose_lock(
    g: dict[str, Any],
    *,
    mode: str,
    expect_mask: bool,
) -> list[str]:
    errs: list[str] = []
    wan = next((n for n in g.values() if n.get("class_type") == "WanAnimateToVideo"), None)
    if wan is None:
        return ["WanAnimateToVideo missing"]
    inn = wan.get("inputs") or {}
    if "pose_video" not in inn:
        errs.append("pose_video not wired")
    if "reference_image" not in inn:
        errs.append("reference_image not wired")
    if any(n.get("class_type") == "MiniMaxH3ReferenceToVideo" for n in g.values()):
        errs.append("H3 R2V node must not be in the pose-lock graph")
    clip = next((n for n in g.values() if n.get("class_type") == "CLIPLoader"), None)
    if clip and (clip.get("inputs") or {}).get("type") != "wan":
        errs.append("CLIPLoader type must be wan")
    if mode == "mix":
        if "background_video" not in inn:
            errs.append("mix needs background_video")
        if expect_mask and "character_mask" not in inn:
            errs.append("mix needs character_mask")
    else:
        if "background_video" in inn or "character_mask" in inn:
            errs.append("move mode must not wire background/mask")
    if not any(n.get("class_type") == "DWPreprocessor" for n in g.values()):
        errs.append("DWPreprocessor missing")
    return errs


def default_mix_prompt(image_name: str) -> str:
    return (
        f"The character from the reference image ({image_name}) performs the motion. "
        "Keep the exact face, hair, body, and costume from the still. "
        "Photoreal live-action, same camera as the source clip, no text."
    )


def even16(n: int) -> int:
    return max(16, int(n) // 16 * 16)


def snap_video_size(width: int, height: int, max_edge: int = 640) -> tuple[int, int]:
    width, height = int(width), int(height)
    if max(width, height) > max_edge:
        scale = max_edge / float(max(width, height))
        width = int(width * scale)
        height = int(height * scale)
    return even16(width), even16(height)


def write_person_mask_videos(
    video_path: str | Path,
    out_dir: str | Path,
    n_people: int = 2,
    max_frames: int = 77,
) -> list[str]:
    """Write one binary mask mp4 per person (left-to-right on first frame).

    Needs ultralytics YOLO-seg. Crossing fighters can swap IDs; inspect the masks.
    Returns filenames relative to out_dir (usually Comfy input/).
    """
    try:
        import cv2  # type: ignore
        from ultralytics import YOLO  # type: ignore
    except ImportError as e:
        raise RuntimeError(
            "人物マスクには ultralytics が必要です。セル9で pip install ultralytics を実行してください。"
        ) from e

    video_path = Path(video_path)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"cannot open {video_path}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 16.0
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    model = YOLO("yolov8n-seg.pt")
    # Track IDs so crossing fighters do not swap left/right every frame.
    stream = model.track(
        source=str(video_path),
        persist=True,
        classes=[0],
        stream=True,
        verbose=False,
        tracker="bytetrack.yaml",
    )
    first_map: dict[Any, int] | None = None
    writers = []
    names: list[str] = []
    n_written = 0
    for result in stream:
        if n_written >= max_frames:
            break
        frame = result.orig_img
        if frame is None:
            continue
        if not writers:
            for i in range(n_people):
                name = f"{video_path.stem}_mask_p{i + 1}.mp4"
                path = out_dir / name
                fourcc = cv2.VideoWriter_fourcc(*"mp4v")
                writers.append(cv2.VideoWriter(str(path), fourcc, fps, (w, h), True))
                names.append(name)
        canvases = [(frame * 0).astype("uint8") for _ in range(n_people)]
        boxes = result.boxes
        if boxes is not None and len(boxes) and boxes.xyxy is not None:
            xyxy = boxes.xyxy.cpu().tolist()
            ids = boxes.id.cpu().tolist() if boxes.id is not None else list(range(len(xyxy)))
            recs = [
                (ids[i], float(xyxy[i][0]), float(xyxy[i][1]), float(xyxy[i][2]), float(xyxy[i][3]))
                for i in range(len(xyxy))
            ]
            if first_map is None:
                first_map = assign_people_left_to_right(recs[:n_people])
            if result.masks is not None:
                for j, rec in enumerate(recs):
                    person_i = first_map.get(rec[0])
                    if person_i is None or person_i >= n_people:
                        continue
                    m = result.masks.data[j].cpu().numpy()
                    m = cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST)
                    canvases[person_i][m > 0.5] = (255, 255, 255)
        for wr, canvas in zip(writers, canvases):
            wr.write(canvas)
        n_written += 1
    for wr in writers:
        wr.release()
    if not n_written or first_map is None:
        raise RuntimeError("最初のフレームで人物を検出できませんでした")
    return names


In [ ]:
%%writefile h3_motion_graphics.py
"""MiniMax H3 I2VA / FL2VA pack for a 10-shot motion-graphics homage.

This is NOT R2V. The original demo's motion is reconstructed from the instruction
sheets as timed shots. Picture 1 is the first frame at 0.00s.

Do not put affiliate URLs in prompts or git. On-screen CTA only; the clickable
link lives on the human's profile.
"""

from __future__ import annotations

from pathlib import Path
from typing import Any

from h3_r2v_core import comfy_media_name, frames

# X @ponzponz15/2091744536716611856: 1280x1440 (8:9), 9.87s, 30fps, one still → video.
# Homage keeps 8:9. 1024x1152 OOMs on A100 40GB I2VA 10s; default is 768x864.
DURATION_S = 10.0
CANVAS_8_9 = (768, 864)
CANVAS_8_9_HIGH = (1024, 1152)
CANVAS_8_9_NATIVE = (1280, 1440)
CANVAS_8_9_MIN = (512, 576)
CANVAS_8_9_LADDER = (CANVAS_8_9_NATIVE, CANVAS_8_9_HIGH, CANVAS_8_9, CANVAS_8_9_MIN)
DEFAULT_FIRST_IMAGE = "Image 1.jpg"

I2VA_HEADER = (
    "For the target video, at 0.00 seconds into the target video, "
    "<Picture 1> (from [Shot 1]) is fully referenced."
)

CHARACTER_LOCK = (
    "photorealistic young Japanese woman in her early 20s, long straight dark brown hair "
    "with soft bangs, gentle warm smile, fair translucent skin, natural makeup, wearing a "
    "white linen sleeveless camisole top with small buttons and matching long flowing "
    "drawstring skirt, barefoot, standing in a traditional Japanese tatami room with shoji "
    "doors and bonsai, soft natural sunlight, highly detailed realistic skin and fabric texture, "
    "pure photorealism"
)

COPY = {
    "main_a": "好きは、仕事になる。",
    "main_b": "未経験から、クリエイターへ。",
    "sub": "プロのスキルが、すぐ見つかる。",
    "learn": "動画・デザイン・AIを実践で学ぶ",
    "card_video": "動画編集",
    "card_video_sub": "ゼロから学べる",
    "card_design": "デザイン",
    "card_design_sub": "想いをカタチに",
    "card_ai": "AI活用",
    "card_ai_sub": "未来の武器になる",
    "timer": "たった1分で無料登録",
    "easy": "カンタン申込み！",
    "cta": "無料で始める",
    "cta_sub": "ココナラでスキルを探す",
    "pr": "広告",
}

FORBIDDEN_IN_PROMPT = (
    "px.a8.net",
    "a8mat=",
    "稼げる",
    "必ず稼",
    "月収",
    "年収",
)


def fl2va_header(duration_s: float = DURATION_S) -> str:
    end = f"{float(duration_s):.2f}"
    return (
        "How the reference pictures align with the target video — "
        "Picture 1 (from Shot 1) aligns with the 0.00-second mark of the target video; "
        f"Picture 2 (from Shot 10) aligns with the {end}-second mark of the target video."
    )


def build_i2va_prompt(*, duration_s: float = DURATION_S, with_last_frame: bool = False) -> str:
    """Official MiniMax H3 I2VA / FL2VA body. On-screen Japanese stays in quotes."""
    d = float(duration_s)
    header = fl2va_header(d) if with_last_frame else I2VA_HEADER
    c = COPY
    body = f"""integrated_multimodal_description: [Shot 1] Live-action, cinematic photorealism, no anime and no illustration. <Picture 1> is the exact first frame. The woman is {CHARACTER_LOCK}. The camera holds a static shot then adds 2.5D parallax with small amplitude at slow speed: hair sways, sunlight on shoji shifts, she blinks once. Identity, clothes, eye color, and room stay locked.
[Shot 2] At 00:01.000, the shot transitions with a soft diagonal wipe as motion-graphic type, not a subtitle, slides in: "{c['main_a']}" appears with outline registration then fill, followed by tracking on "{c['main_b']}". A tiny "{c['pr']}" mark sits in a corner. The previous wipe triggers the type.
[Shot 3] At 00:02.000, the camera cuts to a close-up of her right hand. Fingers move as if drawing; cyan trim-path lines extend from the fingertip and write "{c['main_b'][:3]}" as "未経験" in the air above the tatami. The line motion is caused by the hand.
[Shot 4] At 00:03.000, the shot pulls back with small amplitude at slow speed. Huge tracking type "{c['main_b']}" floats in front of her. Her smile stays the same face from <Picture 1>, with a slightly more hopeful catchlight. Hair continues to sway.
[Shot 5] At 00:04.500, the giant type triggers two clean pop-in cards of copy: "{c['learn']}" and "{c['sub']}". Text is a graphic object in 3D space, not burned-in captions.
[Shot 6] At 00:05.500, three portal cards open in a row, each caused by the previous pop: "{c['card_video']}" / "{c['card_video_sub']}"; "{c['card_design']}" / "{c['card_design_sub']}"; "{c['card_ai']}" / "{c['card_ai_sub']}". Icons look like real objects, not flat anime stickers.
[Shot 7] At 00:06.500, each card expands as a portal window into a photoreal skill-work scene (editing timeline, design canvas, AI interface) while the woman remains the same person from <Picture 1> in the room behind the cards.
[Shot 8] At 00:07.500, a badge scales up: "{c['timer']}" and "{c['easy']}". No income claims. The portal motion triggers the badge.
[Shot 9] At 00:08.500, the camera pulls out with medium amplitude at slow speed to the full advertisement layout. She looks toward the lens with the same gentle smile.
[Shot 10] At 00:09.200, a red CTA button reading "{c['cta']}" scales up and stays locked until {d:.2f} seconds. Secondary type "{c['cta_sub']}" sits under it. The "{c['pr']}" mark remains. Do not drop the CTA. Do not change her face, hair, or clothes.

overall_soundscape: Quiet tatami-room ambience, soft fabric rustle, a faint stylus tick when the trim-path line is drawn, light UI whooshes as cards open, a soft click when the CTA locks.

non_diegetic_music: Sparse warm piano at a moderate tempo with a low pulse that rises slightly into the final button hold, then holds a single resolving chord.
"""
    return header + "\n\n" + body.strip() + "\n"


def resolve_motion_prompt(
    prompt: str | None,
    *,
    duration_s: float = DURATION_S,
    with_last_frame: bool = False,
) -> str:
    """Use the pre-filled cell prompt, or rebuild if empty / last-frame lock is missing."""
    text = (prompt or "").strip()
    if not text:
        return build_i2va_prompt(duration_s=duration_s, with_last_frame=with_last_frame)
    if with_last_frame and "Picture 2" not in text:
        return build_i2va_prompt(duration_s=duration_s, with_last_frame=True)
    return text


def i2va_retry_plans(*, width: int, height: int) -> list[dict[str, Any]]:
    """Smaller 8:9 canvases after OOM. Never drops first_frame or switches to R2V."""
    w = max(32, int(width) // 32 * 32)
    h = max(32, int(height) // 32 * 32)
    plans = [{"width": w, "height": h, "label": f"{w}x{h}"}]
    area = w * h
    for cw, ch in CANVAS_8_9_LADDER:
        if cw * ch < area:
            plans.append({"width": int(cw), "height": int(ch), "label": f"{cw}x{ch}"})
    out: list[dict[str, Any]] = []
    seen: set[tuple[int, int]] = set()
    for p in plans:
        key = (p["width"], p["height"])
        if key in seen:
            continue
        seen.add(key)
        out.append(p)
    return out


def validate_motion_ad_prompt(prompt: str, *, with_last_frame: bool = False) -> list[str]:
    errs: list[str] = []
    p = prompt or ""
    low = p.lower()
    if with_last_frame:
        if "Picture 2" not in p or "aligns with the" not in p:
            errs.append("FL2VA header missing Picture 2 end alignment")
    else:
        if I2VA_HEADER not in p:
            errs.append("I2VA 0.00s Picture 1 header missing")
    if "integrated_multimodal_description:" not in p:
        errs.append("missing integrated_multimodal_description")
    if "overall_soundscape:" not in p:
        errs.append("missing overall_soundscape")
    if "non_diegetic_music:" not in p:
        errs.append("missing non_diegetic_music")
    for i in range(1, 11):
        if f"[Shot {i}]" not in p:
            errs.append(f"missing [Shot {i}]")
    for key in ("main_a", "main_b", "cta", "pr", "card_video", "cta_sub"):
        if COPY[key] not in p:
            errs.append(f"missing on-screen copy {key}")
    if "photoreal" not in low:
        errs.append("photoreal lock missing")
    if "<Picture 1>" not in p:
        errs.append("Picture 1 tag missing")
    for bad in FORBIDDEN_IN_PROMPT:
        if bad.lower() in low:
            errs.append(f"forbidden string in prompt: {bad}")
    return errs


def prefer_fl2v_lora(lora_paths: list[Path], use_lora: bool) -> str | None:
    if not use_lora or not lora_paths:
        return None
    names = [p.name for p in lora_paths if p.suffix.lower() == ".safetensors"]
    fl = [n for n in names if "fl2v" in n.lower() or "fl2va" in n.lower()]
    if fl:
        turbo = [n for n in fl if "turbo" in n.lower()]
        return sorted(turbo or fl, reverse=True)[0]
    return None


def build_i2va_graph(
    *,
    first_image: str,
    last_image: str | None,
    prompt: str,
    unet: str,
    lora_name: str | None,
    lora_strength: float,
    width: int,
    height: int,
    duration_s: float,
    seed: int,
    steps: int,
    filename_prefix: str,
    has_lora_loader: bool = True,
    has_audio_decode: bool = True,
    clip_name: str = "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
    vvae: str = "minimax_h3_video_vae_fp16.safetensors",
    avae: str = "minimax_h3_audio_vae_fp32.safetensors",
) -> dict[str, Any]:
    """MiniMaxH3ImageToVideo: first frame required, last frame optional (CTA lock)."""
    if width % 32 or height % 32:
        raise ValueError(f"H3 width/height must be multiples of 32, got {width}x{height}")
    g: dict[str, Any] = {}
    length = frames(duration_s)
    g["100"] = {
        "class_type": "LoadImage",
        "inputs": {"image": comfy_media_name(first_image)},
    }
    g["1"] = {
        "class_type": "UNETLoader",
        "inputs": {"unet_name": unet, "weight_dtype": "default"},
    }
    model: list[Any] = ["1", 0]
    if lora_name and has_lora_loader:
        g["2"] = {
            "class_type": "LoraLoaderModelOnly",
            "inputs": {
                "model": ["1", 0],
                "lora_name": lora_name,
                "strength_model": float(lora_strength),
            },
        }
        model = ["2", 0]
    g["3"] = {
        "class_type": "CLIPLoader",
        "inputs": {"clip_name": clip_name, "type": "minimax", "device": "default"},
    }
    g["4"] = {"class_type": "VAELoader", "inputs": {"vae_name": vvae}}
    g["5"] = {"class_type": "VAELoader", "inputs": {"vae_name": avae}}
    i_inputs: dict[str, Any] = {
        "clip": ["3", 0],
        "vae": ["4", 0],
        "prompt": prompt,
        "width": int(width),
        "height": int(height),
        "length": length,
        "first_frame": ["100", 0],
    }
    if last_image:
        g["101"] = {
            "class_type": "LoadImage",
            "inputs": {"image": comfy_media_name(last_image)},
        }
        i_inputs["last_frame"] = ["101", 0]
    g["20"] = {"class_type": "MiniMaxH3ImageToVideo", "inputs": i_inputs}
    g["21"] = {"class_type": "RandomNoise", "inputs": {"noise_seed": int(seed)}}
    sampler = "euler" if lora_name else "res_multistep"
    scheduler = "simple" if lora_name else "beta"
    g["22"] = {"class_type": "KSamplerSelect", "inputs": {"sampler_name": sampler}}
    g["23"] = {
        "class_type": "BasicScheduler",
        "inputs": {
            "model": model,
            "scheduler": scheduler,
            "steps": int(steps) if lora_name else max(int(steps), 16),
            "denoise": 1.0,
        },
    }
    g["24"] = {
        "class_type": "BasicGuider",
        "inputs": {"model": model, "conditioning": ["20", 0]},
    }
    g["25"] = {
        "class_type": "SamplerCustomAdvanced",
        "inputs": {
            "noise": ["21", 0],
            "guider": ["24", 0],
            "sampler": ["22", 0],
            "sigmas": ["23", 0],
            "latent_image": ["20", 1],
        },
    }
    g["26"] = {"class_type": "VAEDecode", "inputs": {"samples": ["25", 0], "vae": ["4", 0]}}
    if has_audio_decode:
        g["27"] = {
            "class_type": "VAEDecodeAudio",
            "inputs": {"samples": ["25", 0], "vae": ["5", 0]},
        }
        g["28"] = {
            "class_type": "CreateVideo",
            "inputs": {"images": ["26", 0], "audio": ["27", 0], "fps": 24},
        }
    else:
        g["28"] = {"class_type": "CreateVideo", "inputs": {"images": ["26", 0], "fps": 24}}
    g["29"] = {
        "class_type": "SaveVideo",
        "inputs": {
            "video": ["28", 0],
            "filename_prefix": filename_prefix,
            "format": "auto",
            "codec": "auto",
        },
    }
    return g


def assert_i2va_graph(g: dict[str, Any], *, expect_last: bool) -> list[str]:
    errs: list[str] = []
    node = g.get("20") or {}
    if node.get("class_type") != "MiniMaxH3ImageToVideo":
        errs.append("node 20 must be MiniMaxH3ImageToVideo")
    inn = node.get("inputs") or {}
    if "first_frame" not in inn:
        errs.append("first_frame not wired")
    if expect_last and "last_frame" not in inn:
        errs.append("last_frame not wired")
    if not expect_last and "last_frame" in inn:
        errs.append("I2VA should not wire last_frame")
    if any(n.get("class_type") == "MiniMaxH3ReferenceToVideo" for n in g.values()):
        errs.append("R2V node must not be in the I2VA graph")
    prompt = inn.get("prompt") or ""
    errs.extend(validate_motion_ad_prompt(prompt, with_last_frame=expect_last))
    return errs


In [ ]:
#@title セル8a：作業フォルダへコアファイルを上書き
from pathlib import Path
import sys

DESKTOP = Path(r"C:\Users\ys734\Desktop\minimaxh3")
names = ["h3_r2v_core.py", "pose_motion_lock.py", "h3_motion_graphics.py"]
for name in names:
    src = Path(name)
    if not src.is_file():
        src = Path("/content") / name
    if not src.is_file():
        print("skip missing", name, "（writefile セルを先に実行）")
        continue
    body = src.read_text(encoding="utf-8")
    dests = [
        DESKTOP / name,
        Path("/content") / name,
        Path("/content/drive/MyDrive/minimax-h3-comfyui") / name,
        Path.cwd() / name,
    ]
    seen = set()
    for d in dests:
        try:
            key = str(d.resolve()) if d.exists() or d.parent.exists() else str(d)
            if key in seen:
                continue
            seen.add(key)
            if d.parent == DESKTOP and not DESKTOP.is_dir():
                print("skip (no Windows folder):", d)
                continue
            d.parent.mkdir(parents=True, exist_ok=True)
            d.write_text(body, encoding="utf-8")
            print("overwrote", d)
        except Exception as e:
            print("skip", d, ":", e)

for p in (DESKTOP, Path.cwd(), Path("/content"), Path("/content/drive/MyDrive/minimax-h3-comfyui")):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
try:
    from h3_motion_graphics import build_i2va_prompt
    prompt_body = build_i2va_prompt()
    for d in (
        DESKTOP / "coconala_h3_i2va_prompt.txt",
        Path("/content") / "coconala_h3_i2va_prompt.txt",
        Path("/content/drive/MyDrive/minimax-h3-comfyui") / "coconala_h3_i2va_prompt.txt",
        Path.cwd() / "coconala_h3_i2va_prompt.txt",
    ):
        try:
            if d.parent == DESKTOP and not DESKTOP.is_dir():
                continue
            d.parent.mkdir(parents=True, exist_ok=True)
            d.write_text(prompt_body, encoding="utf-8")
            print("overwrote", d)
        except Exception as e:
            print("skip", d, ":", e)
except Exception as e:
    print("prompt skip:", e)
print("Desktop folder exists:", DESKTOP.is_dir())


In [ ]:
#@title セル8：R2V（参照画像=人物 / 参照動画=モーション）
# ##############################################################################
print("=" * 60)
print(" セル8：R2V — stills=identity, video=motion")
print("=" * 60)
print("⚠ MiniMax H3 R2V はモーションキャプチャではない（参照して再生成）。")
print("  刀・蹴り・激しいカメラを完全模倣するなら、このセルではなく【セル9】Wan Animate Mix。")
print("=" * 60)

import json, os, sys, time, uuid, urllib.request, urllib.error, shutil
from pathlib import Path

# ---------- ユーザー設定 ----------
IMAGE_FILES = "Image 1.jpg,Image 2.jpg"  #@param {type:"string"}
VIDEO_FILES = "0815(1).mp4"  #@param {type:"string"}
PROMPT = ""  #@param {type:"string"}
WIDTH = 960  #@param {type:"integer"}
HEIGHT = 544  #@param {type:"integer"}
DURATION_S = 14  #@param {type:"number"}
STEPS = 4  #@param {type:"integer"}
SEED = 42  #@param {type:"integer"}
USE_LORA = True  #@param {type:"boolean"}
LORA_STRENGTH = 1.0  #@param {type:"number"}
REF_IMAGE_SIZE = "max"  #@param ["max", "match"]
FILENAME_PREFIX = "video/h3_r2v_flex"  #@param {type:"string"}
USE_MOTION = True  #@param {type:"boolean"}
INJECT_ROLE_LOCK = True  #@param {type:"boolean"}
# 既定OFF。線画化すると H3 がモーションを読めなくなることが多い。顔が混ざるときだけ True。
STRIP_VIDEO_IDENTITY = False  #@param {type:"boolean"}
STRIP_MODE = "edges"  #@param ["edges", "blur"]
DRY_RUN = False  #@param {type:"boolean"}
# --------------------------------

DESKTOP = Path(r"C:\Users\ys734\Desktop\minimaxh3")
for p in (DESKTOP, Path.cwd(), Path("/content"), Path("/content/drive/MyDrive/minimax-h3-comfyui")):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from h3_r2v_core import (
    frames, parse_list, prefer_ref2v_lora, finalize_prompt, build_r2v_graph,
    image_ref_key, video_ref_key, assert_graph_identity_motion, prepare_motion_refs,
    comfy_media_name, cap_duration_for_vram, r2v_retry_plans, is_oom_error,
    format_gpu_runtime_note,
)

env = {}
env_path = Path("/content/h3_paths.env")
if env_path.is_file():
    with open(env_path) as f:
        for line in f:
            if "=" in line:
                k, v = line.strip().split("=", 1)
                env[k] = v
else:
    env = {"COMFY_DIR": str(Path.cwd() / "ComfyUI"), "DRIVE_ROOT": str(Path.cwd())}

COMFY_DIR = Path(env.get("COMFY_DIR", "/content/ComfyUI"))
DRIVE_ROOT = env.get("DRIVE_ROOT", "/content/drive/MyDrive/minimax-h3-comfyui")
INP = COMFY_DIR / "input"
PORT = 8188
MAX_IMAGES, MAX_VIDEOS = 9, 3
IMG_EXT = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
VID_EXT = {".mp4", ".mov", ".webm", ".mkv", ".avi"}

def list_media(exts):
    if not INP.exists():
        return []
    return sorted(
        [p for p in INP.rglob("*") if p.is_file() and p.suffix.lower() in exts],
        key=lambda p: str(p.relative_to(INP)).lower(),
    )

def resolve_one(n, kind, known_paths):
    n = (n or "").strip().strip('"').strip("'").lstrip("./")
    if not n:
        raise SystemExit(f"empty {kind} name")
    candidates = [INP / n, INP / Path(n).name]
    stem = Path(n).stem if Path(n).suffix else n
    exts = list(VID_EXT) if kind == "video" else list(IMG_EXT)
    if not Path(n).suffix:
        for e in exts:
            candidates += [INP / f"{n}{e}", INP / f"{stem}{e}"]
    for c in candidates:
        if c.is_file():
            return c
    hits = list(INP.rglob(Path(n).name)) if INP.exists() else []
    if hits:
        return hits[0]
    for p in known_paths:
        if p.stem == stem or p.name == n:
            return p
    available = ", ".join(p.name for p in known_paths[:40]) or "(none)"
    raise SystemExit(f"{kind} not found: {n}\ninput: {INP}\navailable: {available}")

def resolve_names(user_list, auto_paths, limit, kind):
    if user_list is not None and len(user_list) == 0 and kind == "video" and not USE_MOTION:
        return []
    if user_list:
        names = []
        for n in user_list[:limit]:
            p = resolve_one(n, kind, auto_paths)
            try:
                rel = str(p.relative_to(INP)).replace("\\", "/")
            except ValueError:
                rel = p.name
            print(f"  resolved {kind}: {n!r} -> {rel}")
            names.append(rel)
        return names
    return [str(p.relative_to(INP)).replace("\\", "/") for p in auto_paths[:limit]]

all_imgs = list_media(IMG_EXT)
all_vids = list_media(VID_EXT)
img_names = (
    resolve_names(None, all_imgs, MAX_IMAGES, "image")
    if IMAGE_FILES.strip() == ""
    else resolve_names(parse_list(IMAGE_FILES), all_imgs, MAX_IMAGES, "image")
)

if not USE_MOTION or VIDEO_FILES.strip().lower() in ("none", "-"):
    vid_names = []
elif VIDEO_FILES.strip() == "":
    vid_names = resolve_names(None, all_vids, MAX_VIDEOS, "video")
else:
    vid_names = resolve_names(parse_list(VIDEO_FILES), all_vids, MAX_VIDEOS, "video")

print("selected images:", img_names)
print("selected videos (raw):", vid_names)

if USE_MOTION and not vid_names and not DRY_RUN:
    raise SystemExit("USE_MOTION=True なのに動画がありません。VIDEO_FILES と input/ を確認してください。")

if vid_names and STRIP_VIDEO_IDENTITY and not DRY_RUN:
    print("STRIP_VIDEO_IDENTITY=True mode=", STRIP_MODE, "— optional identity strip (can weaken motion)")
    vid_names = prepare_motion_refs(INP, vid_names, enabled=True, mode=STRIP_MODE)
    print("selected videos (proxies):", vid_names)
elif vid_names and STRIP_VIDEO_IDENTITY and DRY_RUN:
    print("DRY_RUN: would strip", vid_names, "to", STRIP_MODE)


def _vram_gb():
    try:
        import torch
        if torch.cuda.is_available():
            return torch.cuda.get_device_properties(0).total_memory / 1024**3
    except Exception:
        pass
    return 40.0

def _host_ram_gb():
    try:
        return os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
    except Exception:
        return None

VRAM_GB = _vram_gb()
try:
    import torch
    _gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""
except Exception:
    _gpu_name = ""
print(format_gpu_runtime_note(VRAM_GB, host_ram_gb=_host_ram_gb(), gpu_name=_gpu_name))
if VRAM_GB >= 70:
    print("A100 ハイメモリGPUとして 14秒プランを優先します。")
if not img_names:
    raise SystemExit("人物画像がありません。IMAGE_FILES を指定してください。")

plans = r2v_retry_plans(
    duration_s=float(DURATION_S),
    ref_image_size=REF_IMAGE_SIZE,
    width=int(WIDTH),
    height=int(HEIGHT),
    n_images=len(img_names),
    has_video=bool(vid_names and USE_MOTION),
    vram_gb=VRAM_GB,
)
if plans[0]["duration_s"] + 0.01 < float(DURATION_S):
    print(
        f"⚠ {DURATION_S}s + REF_IMAGE_SIZE={REF_IMAGE_SIZE} + 参照動画 は "
        f"{VRAM_GB:.0f}GB では OOM しやすいので、先に {plans[0]['duration_s']:.0f}s で回します。"
        " 動画は切らず、長さと参照解像度だけ落とします。"
    )
print("retry plans:", [p["label"] for p in plans])

diff = list((COMFY_DIR / "models/diffusion_models").glob("*ref2va*")) if (COMFY_DIR / "models/diffusion_models").exists() else []
if not diff and not DRY_RUN:
    raise SystemExit("ref2va モデルがありません。セル3 の MODE を both か r2v にするか、セル5 を実行してください。")
unet = diff[0].name if diff else "minimax_h3_ref2va_pruned_int8_convrot.safetensors"
lora_dir = COMFY_DIR / "models/loras"
loras_all = list(lora_dir.glob("*.safetensors")) if lora_dir.exists() else []
lora_name = prefer_ref2v_lora(loras_all, USE_LORA)
if USE_LORA and lora_name and "ref2v" not in lora_name.lower():
    print("WARN: Ref2V turbo が無いので", lora_name, "を使います（FL2V turbo は R2V に不適）")
print("unet", unet, "lora", lora_name)

obj = {}
if not DRY_RUN:
    with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/object_info", timeout=60) as r:
        obj = json.loads(r.read().decode())
    for need in ["UNETLoader", "CLIPLoader", "VAELoader", "MiniMaxH3ReferenceToVideo", "SaveVideo"]:
        if need not in obj:
            raise SystemExit(f"Missing node {need}. ComfyUI を更新するかセル2を再実行してください。")
    if vid_names and "VHS_LoadVideo" not in obj and "LoadVideo" not in obj:
        raise SystemExit("Video loader missing。セル2 の Video Helper Suite 導入を確認してください。")

has_vhs = ("VHS_LoadVideo" in obj) or DRY_RUN
has_lora_loader = ("LoraLoaderModelOnly" in obj) or DRY_RUN
has_audio = ("VAEDecodeAudio" in obj) or DRY_RUN

def post_prompt(g):
    body = {"prompt": g, "client_id": str(uuid.uuid4())}
    data = json.dumps(body).encode()
    req = urllib.request.Request(
        f"http://127.0.0.1:{PORT}/prompt",
        data=data,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=180) as r:
            return json.loads(r.read().decode()), None
    except urllib.error.HTTPError as e:
        return None, f"HTTP {e.code}: {e.read().decode('utf-8', errors='replace')[:4000]}"

def wait_prompt(pid, timeout=3600):
    t0 = time.time()
    while time.time() - t0 < timeout:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/history/{pid}", timeout=60) as r:
            hist = json.loads(r.read().decode())
        entry = hist.get(pid) or {}
        status = entry.get("status") or {}
        if status.get("completed") or entry.get("outputs"):
            return True, entry
        for m in status.get("messages") or []:
            if isinstance(m, list) and m and m[0] == "execution_error":
                return False, m
        time.sleep(5)
    return False, "timeout"

def comfy_free():
    try:
        data = json.dumps({"unload_models": True, "free_memory": True}).encode()
        req = urllib.request.Request(
            f"http://127.0.0.1:{PORT}/free",
            data=data,
            headers={"Content-Type": "application/json"},
            method="POST",
        )
        urllib.request.urlopen(req, timeout=60).read()
        print("ComfyUI /free で VRAM を解放しました")
        time.sleep(3)
    except Exception as e:
        print(" /free skip:", e)

def make_graph(plan):
    prompt = finalize_prompt(PROMPT, img_names, vid_names, plan["duration_s"], inject_role_lock=INJECT_ROLE_LOCK)
    if vid_names and STRIP_VIDEO_IDENTITY:
        prompt = (
            "ANTI-LEAK: The motion reference has been identity-stripped (edges/blur). "
            "It contains NO usable face identity. You MUST invent ZERO faces from the video. "
            "All faces MUST come from <Picture 1>"
            + (" and <Picture 2>" if len(img_names) > 1 else "")
            + " only.\n\n"
            + prompt
        )
    g = build_r2v_graph(
        img_names=img_names,
        vid_names=vid_names if USE_MOTION else [],
        prompt=prompt,
        unet=unet,
        lora_name=lora_name,
        lora_strength=float(LORA_STRENGTH),
        width=int(plan["width"]),
        height=int(plan["height"]),
        duration_s=float(plan["duration_s"]),
        seed=int(SEED),
        steps=int(STEPS),
        filename_prefix=FILENAME_PREFIX,
        ref_image_size=plan["ref_image_size"],
        use_videos=bool(vid_names and USE_MOTION),
        has_vhs=has_vhs,
        has_lora_loader=has_lora_loader,
        has_audio_decode=has_audio,
        object_info=obj,
        motion_max_edge=plan.get("motion_max_edge"),
    )
    errs = assert_graph_identity_motion(
        g,
        expect_images=len(img_names),
        expect_videos=len(vid_names) if (vid_names and USE_MOTION) else 0,
        prompt=prompt,
    )
    if errs:
        raise SystemExit("GRAPH CHECK FAILED (motion/identity wiring):\n- " + "\n- ".join(errs))
    return g, prompt

graph_path = Path("/content/h3_r2v_flex_last_graph.json")
if not graph_path.parent.exists():
    graph_path = Path.cwd() / "h3_r2v_flex_last_graph.json"

if DRY_RUN:
    g, prompt = make_graph(plans[0])
    graph_path.write_text(json.dumps(g, ensure_ascii=False, indent=2), encoding="utf-8")
    print("DRY_RUN graph:", graph_path, "plan:", plans[0]["label"])
    print(prompt[:1200])
    raise SystemExit(0)

last_err = None
payload = None
used_plan = None
for i, plan in enumerate(plans, 1):
    print(f"\n=== try {i}/{len(plans)} {plan['label']} frames={frames(plan['duration_s'])} ===")
    if i > 1:
        comfy_free()
    graph, prompt = make_graph(plan)
    r_in = graph["20"]["inputs"]
    print("  ref_image_size:", r_in.get("ref_image_size"), "length:", r_in.get("length"))
    for j in range(len(img_names)):
        print(f"  wired image[{j}] {comfy_media_name(img_names[j])} -> {image_ref_key(j)}")
    if vid_names and USE_MOTION:
        for vi in range(len(vid_names)):
            print(f"  wired video[{vi}] {comfy_media_name(vid_names[vi])} -> {video_ref_key(vi)}")
            vin = graph["190"]["inputs"]
            print("  VHS", {k: vin.get(k) for k in ("video", "force_size", "custom_width", "frame_load_cap", "force_rate")})
    graph_path.write_text(json.dumps(graph, ensure_ascii=False, indent=2), encoding="utf-8")
    print("graph:", graph_path)
    res, last_err = post_prompt(graph)
    if not (res and "prompt_id" in res):
        print("rejected:", last_err)
        if last_err and is_oom_error(last_err):
            continue
        raise SystemExit(f"R2V rejected (動画なしへフォールバックしません): {last_err}")
    pid = res["prompt_id"]
    print("ACCEPTED", pid)
    ok, payload = wait_prompt(pid)
    if ok:
        used_plan = plan
        break
    last_err = payload
    print("runtime fail:", str(payload)[:500])
    if not is_oom_error(payload):
        raise SystemExit(f"R2V runtime fail (動画なしへフォールバックしません): {payload}")
    print("OOM なので動画は維持したまま、長さ/参照解像度を落として再試行します")

if used_plan is None:
    raise SystemExit(
        "R2V OOM が続きました。セル4 を --highvram で再起動し、DURATION_S=5 / REF_IMAGE_SIZE=match で再実行してください。\n"
        f"last: {last_err}"
    )

print("DONE plan", used_plan["label"], json.dumps((payload or {}).get("outputs"), ensure_ascii=False)[:800])

cands = []
for root in [COMFY_DIR / "output", Path(DRIVE_ROOT) / "output"]:
    if root.exists():
        cands.extend(root.rglob("*.mp4"))
cands.sort(key=lambda p: p.stat().st_mtime, reverse=True)
print("\n最新出力:")
for p in cands[:10]:
    print(" ", p, f"{p.stat().st_size/1e6:.2f}MB")
print("unet", unet, "lora", lora_name)


## セル8 の使い方（R2V）

| やりたいこと | 設定 |
|---|---|
| 人物は参照画像 | `IMAGE_FILES` + `REF_IMAGE_SIZE=max` + ROLE LOCK（自動） |
| 動きは参照動画 | `USE_MOTION=True` + `VIDEO_FILES`。失敗しても動画なしへ落とさない |
| 顔が動画側に混ざる | まず画像を増やす（正面・全身）。それでもダメなら `STRIP_VIDEO_IDENTITY=True` |
| 速い生成 | **Ref2V turbo**（FL2V turbo ではない）、`STEPS=4` |
| 14秒を通す | ランタイム **A100 High Memory（VRAM 80GB）**。セル1で `VRAM GiB: 80` を確認。High-RAM だけでは不可 |
| 40GB のまま | 自動で約6秒。動画は切らない |
| **動きを完全に固定したい** | H3 ではなく **セル9**。`STAGE=preview` で骨格確認 → `STAGE=mix` で人物置換 |

素材:

```
マイドライブ/minimax-h3-comfyui/input/
マイドライブ/minimax-h3-comfyui/models/diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors
マイドライブ/minimax-h3-comfyui/models/loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors
```


## セル9：アクションの動きを骨格で固定する（Wan 2.2 Animate Mix）

H3 R2V は「動画を見てそれっぽく作る」だけ。刀・蹴り・走りの完全コピーはできない。

セル9は別モデル **Wan 2.2 Animate** で:

1. 参考動画から DWPose（骨格＋手）を取る → 動きのロック
2. Mix: 元映像のカメラ・タイミングはそのまま、マスクした人物だけ参照画像の顔と衣装に差し替え
3. 2人の斬り合いなら左の人→Image 1、右の人→Image 2 の2パス

手順: セル2（controlnet_aux）→ セル4再起動 → writefile → セル8c（モデルDL）→ セル9 `STAGE=preview` → 骨格が追従してから `STAGE=mix`

先頭 **約5秒（77フレーム @16fps）** だけ。14秒フルは骨格が通ってからチャンクを足す。交差する斬り合いはマスクが入れ替わることがあるので、preview と mask mp4 を必ず見る。


In [ ]:
#@title セル8c：Wan 2.2 Animate モデルを Drive へ（アクション用）
print("=" * 60)
print(" セル8c：Wan Animate モデル → Drive")
print("=" * 60)

import os, sys
from pathlib import Path

env = {}
with open("/content/h3_paths.env") as f:
    for line in f:
        k, v = line.strip().split("=", 1)
        env[k] = v
DRIVE_MODELS = env["DRIVE_MODELS"]

sys.path.insert(0, "/content")
sys.path.insert(0, str(Path.cwd()))
from pose_motion_lock import WAN_MODELS

MIN_OK = 1_000_000
for key, (name, url, sub) in WAN_MODELS.items():
    folder = os.path.join(DRIVE_MODELS, sub)
    os.makedirs(folder, exist_ok=True)
    path = os.path.join(folder, name)
    if os.path.exists(path) and os.path.getsize(path) > MIN_OK:
        print(f"✓ skip {name}  ({os.path.getsize(path)/1e9:.2f} GB)")
        continue
    print(f"↓ {name}")
    !wget -c --show-progress -O "{path}" "{url}"
    if not os.path.exists(path) or os.path.getsize(path) < MIN_OK:
        if os.path.exists(path):
            os.remove(path)
        raise SystemExit(f"DL 失敗: {name}")
    print(f"✓ {name}  ({os.path.getsize(path)/1e9:.2f} GB)")

print("\nOK → Comfy をセル4で再起動してから【セル9】")
print("（controlnet_aux / clip_vision を足したあとはセル4必須）")


In [ ]:
#@title セル9：動き完全ロック（preview=骨格 / mix=人物差し替え）
print("=" * 60)
print(" セル9：Wan 2.2 Animate — pose lock + identity stills")
print("=" * 60)

import json, os, sys, time, uuid, urllib.request, shutil
from pathlib import Path

IMAGE_FILES = "Image 1.jpg,Image 2.jpg"  #@param {type:"string"}
VIDEO_FILES = "0815(1).mp4"  #@param {type:"string"}
STAGE = "preview"  #@param ["preview", "mix"]
MODE = "mix"  #@param ["mix", "move"]
DURATION_S = 5  #@param {type:"number"}
FPS = 16  #@param {type:"number"}
WIDTH = 640  #@param {type:"integer"}
HEIGHT = 368  #@param {type:"integer"}
STEPS = 4  #@param {type:"integer"}
SEED = 42  #@param {type:"integer"}
GROW_MASK = 28  #@param {type:"integer"}
PROMPT = ""  #@param {type:"string"}
DRY_RUN = False  #@param {type:"boolean"}

DESKTOP = Path(r"C:\Users\ys734\Desktop\minimaxh3")
for p in (DESKTOP, Path.cwd(), Path("/content"), Path("/content/drive/MyDrive/minimax-h3-comfyui")):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from h3_r2v_core import parse_list, comfy_media_name, is_oom_error
from pose_motion_lock import (
    H3_NOT_MOCAP, WAN_MODELS, wan_length, chunk_count, snap_video_size,
    mix_pass_plan, default_mix_prompt, build_pose_preview_graph,
    build_wan_animate_graph, assert_graph_pose_lock, write_person_mask_videos,
)

print(H3_NOT_MOCAP)
print(f"14秒クリップならチャンク数の目安: {chunk_count(14, fps=FPS)} （今は先頭 {wan_length(DURATION_S, fps=FPS)} フレームだけ）")

env = {}
env_path = Path("/content/h3_paths.env")
if env_path.is_file():
    with open(env_path) as f:
        for line in f:
            if "=" in line:
                k, v = line.strip().split("=", 1)
                env[k] = v
else:
    env = {"COMFY_DIR": str(Path.cwd() / "ComfyUI"), "DRIVE_ROOT": str(Path.cwd())}

COMFY_DIR = Path(env.get("COMFY_DIR", "/content/ComfyUI"))
DRIVE_ROOT = Path(env.get("DRIVE_ROOT", "/content/drive/MyDrive/minimax-h3-comfyui"))
INP = COMFY_DIR / "input"
PORT = 8188
IMG_EXT = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
VID_EXT = {".mp4", ".mov", ".webm", ".mkv", ".avi"}

def list_media(exts):
    if not INP.exists():
        return []
    return sorted(
        [p for p in INP.rglob("*") if p.is_file() and p.suffix.lower() in exts],
        key=lambda p: str(p.relative_to(INP)).lower(),
    )

def resolve_one(n, kind, known_paths):
    n = (n or "").strip().strip('"').strip("'").lstrip("./")
    if not n:
        raise SystemExit(f"empty {kind} name")
    candidates = [INP / n, INP / Path(n).name]
    for c in candidates:
        if c.is_file():
            return c
    hits = list(INP.rglob(Path(n).name)) if INP.exists() else []
    if len(hits) == 1:
        return hits[0]
    raise SystemExit(f"{kind} not found: {n}")

images = list_media(IMG_EXT)
videos = list_media(VID_EXT)
img_names = [str(resolve_one(n, "image", images).relative_to(INP)) for n in parse_list(IMAGE_FILES)]
vid_names = [str(resolve_one(n, "video", videos).relative_to(INP)) for n in parse_list(VIDEO_FILES)]
print("images:", img_names)
print("video:", vid_names)
if not vid_names:
    raise SystemExit("VIDEO_FILES が空です。動きの元クリップを指定してください。")
if STAGE == "mix" and MODE == "mix" and not img_names:
    raise SystemExit("Mix には参照画像が必要です。")

length = wan_length(DURATION_S, fps=FPS)
width, height = snap_video_size(WIDTH, HEIGHT, max(WIDTH, HEIGHT))
print(f"length={length}  size={width}x{height}  stage={STAGE} mode={MODE}")

def post_prompt(g):
    body = {"prompt": g, "client_id": str(uuid.uuid4())}
    req = urllib.request.Request(
        f"http://127.0.0.1:{PORT}/prompt",
        data=json.dumps(body).encode(),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=120) as r:
        return json.loads(r.read().decode())

def wait_prompt(pid, timeout=3600):
    t0 = time.time()
    while time.time() - t0 < timeout:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/history/{pid}", timeout=60) as r:
            hist = json.loads(r.read().decode())
        if pid in hist:
            rec = hist[pid]
            st = rec.get("status") or {}
            if st.get("completed") or rec.get("outputs"):
                msgs = (st.get("messages") or [])
                for m in msgs:
                    if m and m[0] == "execution_error":
                        return False, m
                return True, rec
            for m in (st.get("messages") or []):
                if m and m[0] == "execution_error":
                    return False, m
        time.sleep(2)
    return False, "timeout"

obj = {}
if not DRY_RUN:
    with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/object_info", timeout=60) as r:
        obj = json.loads(r.read().decode())
    need = ["WanAnimateToVideo", "DWPreprocessor", "CLIPVisionLoader"]
    miss = [n for n in need if n not in obj]
    if miss:
        raise SystemExit(
            "Comfy にノードがありません: "
            + ", ".join(miss)
            + "\nセル2で controlnet_aux を入れ、セル4を再起動し、Wan Animate モデル（セル8c）を入れてください。"
        )

def run_graph(g, label):
    graph_path = Path("/content") / f"h3_{label.replace(' ', '_')}_graph.json"
    graph_path.write_text(json.dumps(g, ensure_ascii=False, indent=2), encoding="utf-8")
    print("graph:", graph_path)
    if DRY_RUN:
        print("DRY_RUN skip submit")
        return True, None
    res = post_prompt(g)
    if not (res and "prompt_id" in res):
        raise SystemExit(f"prompt rejected: {res}")
    pid = res["prompt_id"]
    print("ACCEPTED", pid, label)
    ok, payload = wait_prompt(pid)
    if not ok:
        if is_oom_error(payload):
            raise SystemExit("OOM。WIDTH/HEIGHT を 512 台、DURATION_S=5 にして再実行。動画は切らない。")
        raise SystemExit(f"runtime fail: {payload}")
    print("DONE", label)
    return True, payload

if STAGE == "preview":
    g = build_pose_preview_graph(
        video_name=vid_names[0],
        filename_prefix="video/pose_preview",
        length=length,
        fps=FPS,
        object_info=obj,
    )
    run_graph(g, "pose_preview")
    print("output/video/pose_preview*.mp4 を見て、骨格が刀・足に付いているか確認。ダメならクリップを切り直す。")
else:
    if MODE == "move":
        img = img_names[0]
        prompt = PROMPT.strip() or default_mix_prompt(img)
        g = build_wan_animate_graph(
            image_name=img,
            video_name=vid_names[0],
            prompt=prompt,
            mode="move",
            mask_name=None,
            width=width,
            height=height,
            length=length,
            fps=FPS,
            seed=SEED,
            steps=STEPS,
            filename_prefix="video/wan_move",
            object_info=obj,
        )
        errs = assert_graph_pose_lock(g, mode="move", expect_mask=False)
        if errs:
            raise SystemExit(errs)
        run_graph(g, "move")
    else:
        if not DRY_RUN:
            import subprocess
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "opencv-python-headless"])
        src_vid = INP / vid_names[0]
        mask_names = write_person_mask_videos(
            src_vid, INP, n_people=max(1, len(img_names)), max_frames=length,
        )
        print("masks:", mask_names)
        jobs = mix_pass_plan(img_names, vid_names[0])
        bg = vid_names[0]
        for job, mask in zip(jobs, mask_names):
            job["background_video"] = bg
            prompt = PROMPT.strip() or default_mix_prompt(job["reference_image"])
            g = build_wan_animate_graph(
                image_name=job["reference_image"],
                video_name=job["background_video"],
                mask_name=mask,
                prompt=prompt,
                mode="mix",
                width=width,
                height=height,
                length=length,
                fps=FPS,
                seed=SEED,
                steps=STEPS,
                filename_prefix=job["filename_prefix"],
                object_info=obj,
                grow_mask=GROW_MASK,
            )
            errs = assert_graph_pose_lock(g, mode="mix", expect_mask=True)
            if errs:
                raise SystemExit(errs)
            print("===", job["label"], "===")
            run_graph(g, f"mix_pass{job['pass_index']+1}")
            # next pass uses this output as the new plate
            out_dir = Path(DRIVE_ROOT) / "output"
            pref = Path(job["filename_prefix"]).name
            cands = sorted(out_dir.rglob(f"{pref}*.mp4"), key=lambda p: p.stat().st_mtime) if out_dir.exists() else []
            if cands:
                dest = INP / f"{pref}.mp4"
                shutil.copy2(cands[-1], dest)
                bg = dest.name
                print("next background:", bg)
        print("Mix 完了。output/video/h3_mix_pass*.mp4 を確認。")


## セル10：1枚から10秒のモーション広告（H3 I2VA）

**ファイル名とプロンプトは下のセルにデフォルトで入っている。** 和室の静止画を `Image 1.jpg` として Drive `input/` に置く。

既定キャンバスは **768×864（8:9）**。1024×1152 は A100 40GB で OOM する。OOM したら同じ縦横比のまま自動で下げて再投入する（first frame は切らない）。

手順: writefile → セル8a → セル3 `MODE=both` → **セル10**。プロンプトは触らなくてよい。


In [ ]:
#@title セル10：I2VA モーション広告（Picture1=0.00秒固定）
print("=" * 60)
print(" セル10：H3 I2VA — first frame lock + 10-shot motion ad")
print("=" * 60)
print("ファイル名とプロンプトはデフォルトで入っている。元デモ動画は配線しない。")
print("Canvas 8:9. Default 768x864 (40GB). Native 1280x1440. OOM なら自動で下げる。")

import json, os, sys, time, uuid, urllib.request, urllib.error
from pathlib import Path

FIRST_IMAGE = "Image 1.jpg"  #@param {type:"string"}
LAST_IMAGE = ""  #@param {type:"string"}
WIDTH = 768  #@param {type:"integer"}
HEIGHT = 864  #@param {type:"integer"}
DURATION_S = 10  #@param {type:"number"}
STEPS = 4  #@param {type:"integer"}
SEED = 42  #@param {type:"integer"}
USE_LORA = True  #@param {type:"boolean"}
LORA_STRENGTH = 1.0  #@param {type:"number"}
FILENAME_PREFIX = "video/h3_i2va_coconala_ad"  #@param {type:"string"}
DRY_RUN = False  #@param {type:"boolean"}

# 空にすると自動生成。触らなくてよい。
PROMPT = r'''For the target video, at 0.00 seconds into the target video, <Picture 1> (from [Shot 1]) is fully referenced.

integrated_multimodal_description: [Shot 1] Live-action, cinematic photorealism, no anime and no illustration. <Picture 1> is the exact first frame. The woman is photorealistic young Japanese woman in her early 20s, long straight dark brown hair with soft bangs, gentle warm smile, fair translucent skin, natural makeup, wearing a white linen sleeveless camisole top with small buttons and matching long flowing drawstring skirt, barefoot, standing in a traditional Japanese tatami room with shoji doors and bonsai, soft natural sunlight, highly detailed realistic skin and fabric texture, pure photorealism. The camera holds a static shot then adds 2.5D parallax with small amplitude at slow speed: hair sways, sunlight on shoji shifts, she blinks once. Identity, clothes, eye color, and room stay locked.
[Shot 2] At 00:01.000, the shot transitions with a soft diagonal wipe as motion-graphic type, not a subtitle, slides in: "好きは、仕事になる。" appears with outline registration then fill, followed by tracking on "未経験から、クリエイターへ。". A tiny "広告" mark sits in a corner. The previous wipe triggers the type.
[Shot 3] At 00:02.000, the camera cuts to a close-up of her right hand. Fingers move as if drawing; cyan trim-path lines extend from the fingertip and write "未経験" as "未経験" in the air above the tatami. The line motion is caused by the hand.
[Shot 4] At 00:03.000, the shot pulls back with small amplitude at slow speed. Huge tracking type "未経験から、クリエイターへ。" floats in front of her. Her smile stays the same face from <Picture 1>, with a slightly more hopeful catchlight. Hair continues to sway.
[Shot 5] At 00:04.500, the giant type triggers two clean pop-in cards of copy: "動画・デザイン・AIを実践で学ぶ" and "プロのスキルが、すぐ見つかる。". Text is a graphic object in 3D space, not burned-in captions.
[Shot 6] At 00:05.500, three portal cards open in a row, each caused by the previous pop: "動画編集" / "ゼロから学べる"; "デザイン" / "想いをカタチに"; "AI活用" / "未来の武器になる". Icons look like real objects, not flat anime stickers.
[Shot 7] At 00:06.500, each card expands as a portal window into a photoreal skill-work scene (editing timeline, design canvas, AI interface) while the woman remains the same person from <Picture 1> in the room behind the cards.
[Shot 8] At 00:07.500, a badge scales up: "たった1分で無料登録" and "カンタン申込み！". No income claims. The portal motion triggers the badge.
[Shot 9] At 00:08.500, the camera pulls out with medium amplitude at slow speed to the full advertisement layout. She looks toward the lens with the same gentle smile.
[Shot 10] At 00:09.200, a red CTA button reading "無料で始める" scales up and stays locked until 10.00 seconds. Secondary type "ココナラでスキルを探す" sits under it. The "広告" mark remains. Do not drop the CTA. Do not change her face, hair, or clothes.

overall_soundscape: Quiet tatami-room ambience, soft fabric rustle, a faint stylus tick when the trim-path line is drawn, light UI whooshes as cards open, a soft click when the CTA locks.

non_diegetic_music: Sparse warm piano at a moderate tempo with a low pulse that rises slightly into the final button hold, then holds a single resolving chord.
'''

DESKTOP = Path(r"C:\Users\ys734\Desktop\minimaxh3")
for p in (DESKTOP, Path.cwd(), Path("/content"), Path("/content/drive/MyDrive/minimax-h3-comfyui")):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from h3_r2v_core import is_oom_error, frames
from h3_motion_graphics import (
    build_i2va_prompt, validate_motion_ad_prompt, build_i2va_graph,
    assert_i2va_graph, prefer_fl2v_lora, resolve_motion_prompt, i2va_retry_plans,
)

env = {}
env_path = Path("/content/h3_paths.env")
if env_path.is_file():
    with open(env_path) as f:
        for line in f:
            if "=" in line:
                k, v = line.strip().split("=", 1)
                env[k] = v
else:
    env = {"COMFY_DIR": str(Path.cwd() / "ComfyUI"), "DRIVE_ROOT": str(Path.cwd())}

COMFY_DIR = Path(env.get("COMFY_DIR", "/content/ComfyUI"))
INP = COMFY_DIR / "input"
PORT = 8188

def resolve_one(n):
    n = (n or "").strip().strip('"').strip("'").lstrip("./")
    if not n:
        raise SystemExit("FIRST_IMAGE が空です。和室の女の子の静止画を Image 1.jpg として置いてください。")
    for c in [INP / n, INP / Path(n).name]:
        if c.is_file():
            return c
    hits = list(INP.rglob(Path(n).name)) if INP.exists() else []
    if len(hits) == 1:
        return hits[0]
    raise SystemExit(f"画像が見つかりません: {n}  → Drive input/ に {n} として置いてください")

first = str(resolve_one(FIRST_IMAGE).relative_to(INP))
last = str(resolve_one(LAST_IMAGE).relative_to(INP)) if LAST_IMAGE.strip() else None
print("Picture 1 (0.00s):", first)
print("Picture 2 (end CTA):", last or "(none — I2VA)")

prompt = resolve_motion_prompt(PROMPT, duration_s=float(DURATION_S), with_last_frame=bool(last))
errs = validate_motion_ad_prompt(prompt, with_last_frame=bool(last))
if errs:
    raise SystemExit(errs)
print("--- PROMPT ---")
print(prompt)
print("length frames", frames(DURATION_S))

obj = {}
if not DRY_RUN:
    with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/object_info", timeout=60) as r:
        obj = json.loads(r.read().decode())
    if "MiniMaxH3ImageToVideo" not in obj:
        raise SystemExit("MiniMaxH3ImageToVideo がありません。ComfyUI を最新化してセル4を再起動してください。")

diff = list((COMFY_DIR / "models/diffusion_models").glob("*fl2va*")) if (COMFY_DIR / "models/diffusion_models").exists() else []
if not diff and not DRY_RUN:
    raise SystemExit("fl2va がありません。セル3 MODE=both を実行してください。")
unet = diff[0].name if diff else "minimax_h3_fl2va_pruned_int8_convrot.safetensors"
lora_paths = list((COMFY_DIR / "models/loras").glob("*.safetensors")) if (COMFY_DIR / "models/loras").exists() else []
lora = prefer_fl2v_lora(lora_paths, USE_LORA)
print("unet", unet, "lora", lora)

plans = i2va_retry_plans(width=int(WIDTH), height=int(HEIGHT))
print("retry plans:", [p["label"] for p in plans])

def make_graph(plan):
    g = build_i2va_graph(
        first_image=first,
        last_image=last,
        prompt=prompt,
        unet=unet,
        lora_name=lora,
        lora_strength=float(LORA_STRENGTH),
        width=int(plan["width"]),
        height=int(plan["height"]),
        duration_s=float(DURATION_S),
        seed=int(SEED),
        steps=int(STEPS),
        filename_prefix=FILENAME_PREFIX,
        has_lora_loader=("LoraLoaderModelOnly" in obj) or DRY_RUN,
        has_audio_decode=("VAEDecodeAudio" in obj) or DRY_RUN,
    )
    g_errs = assert_i2va_graph(g, expect_last=bool(last))
    if g_errs:
        raise SystemExit(g_errs)
    return g

def post_prompt(g):
    body = {"prompt": g, "client_id": str(uuid.uuid4())}
    req = urllib.request.Request(
        f"http://127.0.0.1:{PORT}/prompt",
        data=json.dumps(body).encode(),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=180) as r:
            return json.loads(r.read().decode()), None
    except urllib.error.HTTPError as e:
        return None, f"HTTP {e.code}: {e.read().decode('utf-8', errors='replace')[:4000]}"

def wait_prompt(pid, timeout=3600):
    t0 = time.time()
    while time.time() - t0 < timeout:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/history/{pid}", timeout=60) as r:
            hist = json.loads(r.read().decode())
        entry = hist.get(pid) or {}
        status = entry.get("status") or {}
        if status.get("completed") or entry.get("outputs"):
            return True, entry
        for m in status.get("messages") or []:
            if isinstance(m, list) and m and m[0] == "execution_error":
                return False, m
        time.sleep(2)
    return False, "timeout"

def comfy_free():
    try:
        data = json.dumps({"unload_models": True, "free_memory": True}).encode()
        req = urllib.request.Request(
            f"http://127.0.0.1:{PORT}/free",
            data=data,
            headers={"Content-Type": "application/json"},
            method="POST",
        )
        urllib.request.urlopen(req, timeout=60).read()
        print("ComfyUI /free で VRAM を解放しました")
        time.sleep(3)
    except Exception as e:
        print(" /free skip:", e)

graph_path = Path("/content/h3_i2va_ad_graph.json")
last_err = None
ok_entry = None
used = None
for plan in plans:
    g = make_graph(plan)
    graph_path.write_text(json.dumps(g, ensure_ascii=False, indent=2), encoding="utf-8")
    print("try", plan["label"], "graph", graph_path)
    if DRY_RUN:
        print("DRY_RUN skip")
        used = plan
        break
    res, err = post_prompt(g)
    if err:
        last_err = err
        if is_oom_error(err):
            print("OOM on submit", plan["label"], "→ 次の8:9キャンバス")
            comfy_free()
            continue
        raise SystemExit(err)
    if not (res and "prompt_id" in res):
        raise SystemExit(res)
    pid = res["prompt_id"]
    print("ACCEPTED", pid, plan["label"])
    ok, payload = wait_prompt(pid)
    if ok:
        ok_entry = payload
        used = plan
        print("DONE", json.dumps((payload or {}).get("outputs"), ensure_ascii=False)[:600])
        break
    last_err = payload
    if is_oom_error(payload):
        print("OOM", plan["label"], "→ 次の8:9キャンバス（first frame は維持）")
        comfy_free()
        continue
    raise SystemExit(payload)
else:
    raise SystemExit(last_err or "I2VA failed")

print("used canvas", (used or {}).get("label"))
print("CTA と顔が最後まで残っているか確認。URL は動画に出さずプロフィールへ。")
